# The Anatomy of AI · Lecture 20 · Attention — the notebook

In Lecture 19, GPT-2 read *It was a warm, sunny day, so we went for a* … and said **walk** (58%). This lecture opened
the part of each block that lets one word use the words before it: **attention**. Here you take it apart yourself:

1. **which earlier words matter**: hide any set of words from attention (all 4,096 sets) and watch walk;
2. **one head, by hand**: block 11, head 12, from its query, keys and values to what it writes back;
3. **all 144 heads**: where they look, a head that looks one word back, the attention sink, switching heads off;
4. **smaller models**: the real next-word counts in 100 million words, and transformers with 0 and 1 blocks that you
   train yourself;
5. **pictures as tokens**: where a vision transformer's summary token looks.

Blocks and heads are counted from 1, as on the plates; papers count from 0 (block 11, head 12 is their *L10H11*).
Every key number is checked against the lecture's own: a ✓ means you got what the lecture got.

It takes about **5 minutes** on free Colab with a GPU (*Runtime → Change runtime type → T4 GPU*).

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3", "pyarrow==18.1.0", "regex==2024.11.6", "transformers==4.48.3")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

## GPT-2 small, as Lecture 19 built it
The next cell is Lecture 19's notebook in one piece: GPT-2's byte-pair tokenizer from its own two files, the weights
file read by hand, and the forward pass `run()`, which keeps every intermediate number (each head's queries, keys,
values, attention pattern and output). `run(ids, allow=…)` lets you forbid some words from looking at others;
`run(ids, zero_heads=…)` switches heads off. `lens()` reads a running total as words.

In [ ]:
# GPT-2 small, exactly as Lecture 19 built it: no library model, only its files from Hugging Face (no key needed).
import regex as re
HF = "https://huggingface.co/openai-community/gpt2/resolve/main/"
fetch(HF + "vocab.json", "vocab.json", "GPT-2's token list")
fetch(HF + "merges.txt", "merges.txt", "GPT-2's 50,000 merges")
fetch(HF + "model.safetensors", "gpt2.safetensors", "GPT-2 small's weights")

# --- the tokenizer: bytes -> printable characters, a regex split, then the merges in rank order (Lecture 19, Plate VI)
VOCAB = json.load(open("vocab.json", encoding="utf-8"))
ID2TOK = {i: t for t, i in VOCAB.items()}
RANK = {tuple(m.split()): r for r, m in enumerate(l for l in open("merges.txt", encoding="utf-8").read().split("\n")[1:] if l)}
_bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
_cs = _bs + [256 + n for n in range(256 - len(_bs))]
_bs = _bs + [b for b in range(256) if b not in _bs]
B2U = dict(zip(_bs, map(chr, _cs))); U2B = {u: b for b, u in B2U.items()}
SPLIT = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
_seen = {}

def bpe(piece):
    if piece not in _seen:
        w = list(piece)
        while len(w) > 1:
            r, i = min((RANK.get(p, 1 << 30), i) for i, p in enumerate(zip(w, w[1:])))
            if r == 1 << 30:
                break
            a, b, out, j = w[i], w[i + 1], [], 0
            while j < len(w):
                if j < len(w) - 1 and w[j] == a and w[j + 1] == b:
                    out.append(a + b); j += 2
                else:
                    out.append(w[j]); j += 1
            w = out
        _seen[piece] = w
    return _seen[piece]

def encode(text):
    return [VOCAB[t] for m in SPLIT.findall(text) for t in bpe("".join(B2U[b] for b in m.encode("utf-8")))]

def decode(ids):
    return bytes(U2B[ch] for i in ids for ch in ID2TOK[int(i)]).decode("utf-8", errors="replace")

# --- the weights: a safetensors file is an 8-byte length, a JSON table of contents, then the raw numbers
def read_safetensors(path):
    raw = open(path, "rb").read()
    n = int.from_bytes(raw[:8], "little")
    head = json.loads(raw[8:8 + n])
    out = {}
    for name, h in head.items():
        if name != "__metadata__" and not name.endswith((".attn.bias", ".attn.masked_bias")):   # masks, not weights
            a, b = h["data_offsets"]
            out[name] = torch.from_numpy(np.frombuffer(raw, dtype=np.float32, count=(b - a) // 4, offset=8 + n + a)
                                         .reshape(h["shape"]).copy()).to(DEV)
    return out

SD = read_safetensors("gpt2.safetensors")
WTE, WPE = SD["wte.weight"], SD["wpe.weight"]
print(f"GPT-2 small: {sum(v.numel() for v in SD.values()):,} weights")

# --- the forward pass (Lecture 19, Plates XIII-XXI; attention as Lecture 20 takes it apart)
def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    return (x - mu) / torch.sqrt(((x - mu) ** 2).mean(-1, keepdim=True) + eps) * g + b

def gelu(x):                                   # GPT-2's smooth gate (the tanh form)
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

@torch.no_grad()
def run(ids, allow=None, zero_heads=()):
    """One pass of GPT-2 over a list of token ids. Returns every intermediate number in a dict.
    allow: optional (T, T) True/False table of which earlier words each word may look at (on top of 'no peeking ahead').
    zero_heads: (block, head) pairs, counted from 0, whose output is set to zero."""
    ids = torch.as_tensor(ids, device=DEV)
    T = len(ids)
    c = {"tok": WTE[ids], "pos": WPE[:T]}
    x = c["tok"] + c["pos"]
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=DEV))
    if allow is not None:
        mask = mask & allow
    keep = {k: [] for k in ("resid", "ln1", "q", "k", "v", "scores", "pattern", "z", "head_out", "attn_out", "mlp_post", "mlp_out")}
    for l in range(12):
        p = lambda n: SD[f"h.{l}.{n}"]
        keep["resid"].append(x)
        h = layer_norm(x, p("ln_1.weight"), p("ln_1.bias"))
        q, k, v = (h @ p("attn.c_attn.weight") + p("attn.c_attn.bias")).split(768, -1)
        q, k, v = (t.view(T, 12, 64).transpose(0, 1) for t in (q, k, v))            # 12 heads x T words x 64
        s = (q @ k.transpose(-1, -2) / 8).masked_fill(~mask, float("-inf"))          # 8 = the square root of 64
        a = torch.softmax(s, -1)
        z = a @ v
        ho = torch.einsum("htd,hdm->htm", z, p("attn.c_proj.weight").view(12, 64, 768))
        for zl, zh in zero_heads:
            if zl == l:
                ho[zh] = 0
        att = ho.sum(0) + p("attn.c_proj.bias")
        x = x + att
        m_in = layer_norm(x, p("ln_2.weight"), p("ln_2.bias")) @ p("mlp.c_fc.weight") + p("mlp.c_fc.bias")
        post = gelu(m_in)
        mo = post @ p("mlp.c_proj.weight") + p("mlp.c_proj.bias")
        x = x + mo
        for key, val in zip(("ln1", "q", "k", "v", "scores", "pattern", "z", "head_out", "attn_out", "mlp_post", "mlp_out"),
                            (h, q, k, v, s, a, z, ho, att, post, mo)):
            keep[key].append(val)
    keep["resid"].append(x)
    c.update({k: torch.stack(v) for k, v in keep.items()})
    c["final"] = layer_norm(x, SD["ln_f.weight"], SD["ln_f.bias"])
    c["logits"] = c["final"] @ WTE.T                 # the scores: the final total against every word's arrow
    return c

def lens(r):
    """The logit lens: read any running total through the final LayerNorm and the word table."""
    return layer_norm(r, SD["ln_f.weight"], SD["ln_f.bias"]) @ WTE.T

In [ ]:
SENTENCE = "It was a warm, sunny day, so we went for a"
ids = encode(SENTENCE)
WORDS = [decode([i]) for i in ids]
T, WALK = len(ids), encode(" walk")[0]
c = run(ids)
P_WALK = torch.softmax(c["logits"][-1], -1)[WALK].item()
print(" | ".join(WORDS), f"→ walk {100 * P_WALK:.1f}%")
check("walk, with nothing hidden", P_WALK, 0.57975, tol=1e-4)

def p_walk(**kw):
    return torch.softmax(run(ids, **kw)["logits"][-1], -1)[WALK].item()

## Part 1 · Which earlier words matter?

### 1. The last word, alone (Plate I)
Attention is the only place where one word's running total can take anything from another word's. So we can test what
the last word needs by forbidding it to look. `allow[i, j]` says whether word *i* may look at word *j* (always only
earlier words: no peeking ahead). Here every word may see only the last *k* words before it, plus itself.

One word stays visible throughout: the first, *It*. Most heads park their spare attention there (Plate XVI, the
*attention sink*); hiding it breaks the model for a reason unrelated to meaning.

In [ ]:
i = torch.arange(T, device=DEV)
eye = torch.eye(T, dtype=torch.bool, device=DEV)
sink = torch.zeros(T, T, dtype=torch.bool, device=DEV); sink[:, 0] = True       # "It" stays visible
window = []
for k in range(T):
    allow = ((i[:, None] - i[None, :]) <= k) | eye | sink                         # only the last k words, plus itself
    window.append(p_walk(allow=allow))
    print(f"sees the last {k:2} words: walk {100 * window[-1]:6.2f}%   …{' '.join(WORDS[max(1, T - 1 - k):]).strip()}")
check("the last word alone: walk", window[0], 0.00034, tol=2e-5)
check("with the last five words: walk", window[5], 0.57971, tol=2e-4)
check("every window (largest difference)", max(abs(a - b) for a, b in zip(window, [0.00034, 0.16447, 0.21497, 0.32031, 0.49438, 0.57971, 0.58473, 0.5727, 0.57975, 0.57438, 0.57793, 0.57975, 0.57975])), 0, tol=2e-4)
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.plot(range(T), window, color=INK, marker="o", ms=4)
ax.set_xlabel("how many words back the last word may look"); ax.set_ylabel("chance of walk"); plt.show()

### 2. Which words matter? (Plate II)
Now hide any *set* of words: no later word may look at them, in any block. There are 2¹² = 4,096 sets of the first
twelve words (the last word always sees itself). Run them all.

In [ ]:
t = time.time()
hide = []
for m in range(2 ** (T - 1)):                                    # bit j of m set = word j hidden
    allow = torch.ones(T, T, dtype=torch.bool, device=DEV)
    for j in range(T - 1):
        if m >> j & 1:
            allow[j + 1:, j] = False
    hide.append(p_walk(allow=allow | eye))
print(f"4,096 runs in {time.time() - t:.0f} s")
check("all 4,096 sets of hidden words (largest difference)", max(abs(a - b) for a, b in zip(hide, [0.57975, 0.00205, 0.55859, 0.00255, 0.609, 0.00018, 0.47508, 0.00108, 0.57454, 0.0001, 0.55357, 0.00041, 0.56349, 0.00029, 0.49163, 7e-05, 0.55489, 0.0035, 0.53266, 0.00271, 0.58858, 0.00041, 0.46451, 0.00095, 0.56661, 0.00101, 0.54666, 0.00103, 0.60342, 0.00068, 0.51076, 0.0007, 0.57119, 0.00093, 0.54166, 0.00239, 0.61666, 0.00027, 0.44718, 0.00137, 0.38115, 7e-05, 0.29646, 1e-05, 0.38747, 2e-05, 0.32432, 0.0004, 0.46323, 0.00255, 0.45506, 0.00151, 0.54529, 0.00065, 0.4256, 0.00033, 0.51111, 0.00035, 0.36218, 0.00058, 0.51991, 0.00065, 0.36968, 0.00077, 0.56604, 0.00218, 0.51142, 0.00174, 0.59118, 0.00012, 0.41626, 0.00046, 0.5257, 0.00023, 0.47688, 5e-05, 0.53035, 0.00025, 0.43327, 0.00051, 0.55728, 0.0023, 0.52412, 0.00444, 0.57043, 0.0005, 0.4339, 0.00019, 0.56917, 0.00062, 0.52913, 0.00148, 0.59974, 0.00069, 0.4831, 0.00024, 0.53243, 0.00104, 0.45101, 0.00097, 0.58521, 6e-05, 0.36277, 0.00045, 0.17118, 4e-05, 0.19836, 1e-05, 0.22859, 0.00019, 0.2, 0.00086, 0.49884, 0.00042, 0.41977, 0.00054, 0.56155, 0.00069, 0.33771, 0.00221, 0.19318, 3e-05, 0.18313, 2e-05, 0.20999, 0.00164, 0.17467, 0.00277, 0.5423, 0.0015, 0.52522, 0.00181, 0.58779, 8e-05, 0.46228, 0.00146, 0.52908, 0.00024, 0.5094, 0.00017, 0.53009, 0.00016, 0.46412, 0.00022, 0.52322, 0.00064, 0.5106, 0.00369, 0.5591, 0.00234, 0.4569, 0.00257, 0.52942, 0.00034, 0.524, 0.00268, 0.57162, 0.00079, 0.49569, 0.00225, 0.52745, 0.00072, 0.51947, 0.00366, 0.60161, 8e-05, 0.44604, 0.00271, 0.3606, 0.00016, 0.28534, 7e-05, 0.33873, 4e-05, 0.31143, 0.00079, 0.42381, 0.00205, 0.4372, 0.00652, 0.50451, 0.00094, 0.41345, 0.00208, 0.43404, 0.00015, 0.36964, 0.00021, 0.44589, 5e-05, 0.34571, 0.00601, 0.5552, 0.00163, 0.50827, 0.00161, 0.58801, 0.00017, 0.43067, 0.00232, 0.50955, 0.00026, 0.45478, 0.00063, 0.50049, 0.00071, 0.42342, 0.001, 0.52439, 0.00095, 0.49843, 0.00289, 0.54475, 0.00233, 0.42506, 0.00013, 0.5274, 0.00027, 0.495, 0.00133, 0.55785, 0.00124, 0.46595, 0.00027, 0.50182, 0.00057, 0.42302, 0.0022, 0.56026, 0.00011, 0.32992, 0.00249, 0.19218, 7e-05, 0.1722, 1e-05, 0.20264, 0.00098, 0.16737, 0.00167, 0.48117, 0.00145, 0.40511, 0.00205, 0.52028, 0.0008, 0.34591, 0.00015, 0.25348, 0.00015, 0.2107, 4e-05, 0.23075, 2e-05, 0.21292, 0.0, 0.55252, 0.00141, 0.48031, 0.00054, 0.57959, 4e-05, 0.41895, 0.00086, 0.52371, 5e-05, 0.46262, 1e-05, 0.4893, 3e-05, 0.41267, 1e-05, 0.51819, 0.00072, 0.45287, 0.00077, 0.55061, 1e-05, 0.39575, 0.00044, 0.52292, 9e-05, 0.45436, 0.00016, 0.56088, 0.0001, 0.42508, 0.00071, 0.55143, 0.00125, 0.47517, 0.00028, 0.59552, 2e-05, 0.40919, 0.00144, 0.2822, 1e-05, 0.23861, 1e-05, 0.31673, 1e-05, 0.28523, 6e-05, 0.42452, 0.00081, 0.38215, 0.00015, 0.50726, 5e-05, 0.36573, 0.00016, 0.42466, 0.00013, 0.30914, 0.00012, 0.42523, 7e-05, 0.3152, 9e-05, 0.54984, 0.00086, 0.47156, 4e-05, 0.56833, 1e-05, 0.36333, 0.00066, 0.48991, 6e-05, 0.41981, 1e-05, 0.45094, 1e-05, 0.35791, 6e-05, 0.52687, 0.00039, 0.46413, 0.00159, 0.54537, 5e-05, 0.3662, 0.00024, 0.56701, 8e-05, 0.49156, 0.00025, 0.58089, 0.00018, 0.40259, 0.00013, 0.55962, 0.00019, 0.4308, 0.00037, 0.55424, 0.0, 0.33089, 0.00022, 0.14023, 2e-05, 0.16116, 1e-05, 0.20228, 0.0, 0.17182, 0.00015, 0.52, 0.00016, 0.43246, 0.0002, 0.5598, 2e-05, 0.30983, 0.00161, 0.19085, 1e-05, 0.14903, 2e-05, 0.18902, 1e-05, 0.15172, 0.00052, 0.37918, 0.00163, 0.3877, 0.00026, 0.46797, 5e-05, 0.35603, 0.0058, 0.3587, 0.0002, 0.35432, 1e-05, 0.40192, 5e-05, 0.34007, 0.00031, 0.40407, 0.00012, 0.40792, 0.00024, 0.45608, 2e-05, 0.3681, 0.00301, 0.39694, 8e-05, 0.39514, 0.00036, 0.4556, 3e-05, 0.38416, 0.00155, 0.38716, 0.00158, 0.35097, 0.00025, 0.5037, 3e-05, 0.36672, 0.00389, 0.21762, 0.00014, 0.17589, 1e-05, 0.2979, 5e-05, 0.26448, 0.00034, 0.34313, 0.00088, 0.33881, 0.0002, 0.4185, 0.00017, 0.32763, 0.00269, 0.25365, 0.00046, 0.25369, 0.0003, 0.29271, 5e-05, 0.27432, 0.00066, 0.16986, 0.00134, 0.20412, 0.00033, 0.33012, 3e-05, 0.21552, 0.00217, 0.07323, 0.00031, 0.11515, 3e-05, 0.16785, 0.0003, 0.14773, 0.001, 0.20338, 0.00023, 0.25801, 0.00119, 0.42488, 0.0002, 0.29597, 0.00169, 0.13687, 0.00013, 0.18005, 0.00133, 0.41846, 0.00022, 0.34176, 0.00079, 0.52998, 0.00016, 0.4524, 0.00022, 0.56847, 0.0, 0.31928, 9e-05, 0.19583, 1e-05, 0.15323, 1e-05, 0.17827, 0.0, 0.13472, 0.0001, 0.12366, 0.00056, 0.11719, 0.00087, 0.38329, 0.00016, 0.20315, 0.0033, 0.12799, 8e-05, 0.11038, 0.00036, 0.12724, 3e-05, 0.12536, 2e-05, 0.43377, 0.00072, 0.13052, 0.00014, 0.5291, 3e-05, 0.14418, 0.00012, 0.36918, 1e-05, 0.06598, 9e-05, 0.38635, 5e-05, 0.06501, 1e-05, 0.40264, 6e-05, 0.12665, 0.00083, 0.48228, 2e-05, 0.13631, 5e-05, 0.41146, 3e-05, 0.08734, 0.00056, 0.50641, 0.00014, 0.08943, 0.0003, 0.31084, 0.00022, 0.04734, 9e-05, 0.44722, 2e-05, 0.04447, 0.00012, 0.03664, 0.0, 0.00818, 1e-05, 0.03554, 0.0, 0.01442, 0.00014, 0.33746, 0.00034, 0.08649, 8e-05, 0.45722, 2e-05, 0.10145, 2e-05, 0.20237, 0.00013, 0.02147, 4e-05, 0.2949, 2e-05, 0.02685, 0.00019, 0.29863, 0.00068, 0.04702, 0.00025, 0.44193, 2e-05, 0.05806, 9e-05, 0.09832, 7e-05, 0.01697, 4e-05, 0.16934, 4e-05, 0.02912, 0.00014, 0.30559, 3e-05, 0.06672, 0.00189, 0.42101, 2e-05, 0.06873, 1e-05, 0.26969, 0.0, 0.03415, 0.00033, 0.42718, 4e-05, 0.03757, 5e-05, 0.12282, 4e-05, 0.01632, 4e-05, 0.22892, 0.0, 0.01855, 3e-05, 0.00498, 0.0, 0.00517, 1e-05, 0.00658, 0.0, 0.00536, 2e-05, 0.16861, 1e-05, 0.02647, 1e-05, 0.32795, 0.0, 0.02411, 3e-05, 0.00668, 0.0, 0.00411, 1e-05, 0.00465, 0.0, 0.00298, 7e-05, 0.33389, 0.00011, 0.05982, 0.00105, 0.48943, 8e-05, 0.13682, 6e-05, 0.23322, 2e-05, 0.02732, 0.00017, 0.2082, 0.00014, 0.02732, 0.00015, 0.43673, 0.00015, 0.13172, 0.00146, 0.4832, 0.00013, 0.21854, 0.00043, 0.44377, 5e-05, 0.04471, 0.00077, 0.50585, 5e-05, 0.08874, 0.00062, 0.0713, 9e-05, 0.0159, 0.00029, 0.19981, 0.0, 0.01924, 1e-05, 0.00713, 1e-05, 0.00405, 0.0, 0.00643, 0.0, 0.00624, 6e-05, 0.35972, 0.00071, 0.04486, 0.00032, 0.43766, 6e-05, 0.12928, 0.00191, 0.0522, 6e-05, 0.00644, 5e-05, 0.22214, 2e-05, 0.00568, 0.00088, 0.13101, 4e-05, 0.01738, 0.00036, 0.41968, 1e-05, 0.05471, 2e-05, 0.03282, 0.0, 0.00985, 2e-05, 0.09863, 6e-05, 0.0161, 5e-05, 0.38571, 6e-05, 0.06595, 7e-05, 0.44395, 7e-05, 0.14741, 2e-05, 0.31725, 1e-05, 0.01402, 6e-05, 0.45226, 2e-05, 0.06312, 6e-05, 0.15438, 3e-05, 0.02452, 4e-05, 0.30591, 0.0, 0.02272, 0.0, 0.00586, 0.0, 0.00339, 0.0, 0.00388, 0.0, 0.00266, 2e-05, 0.22535, 3e-05, 0.00912, 4e-05, 0.38371, 1e-05, 0.03985, 0.0, 0.00457, 3e-05, 0.00166, 0.0, 0.00401, 0.0, 0.00107, 1e-05, 0.29688, 0.00205, 0.01572, 6e-05, 0.52477, 1e-05, 0.11865, 6e-05, 0.18868, 4e-05, 0.01921, 0.0, 0.28561, 1e-05, 0.08066, 0.0, 0.48273, 0.00013, 0.10442, 0.00042, 0.52734, 1e-05, 0.24745, 1e-05, 0.49043, 1e-05, 0.08046, 0.00015, 0.55332, 2e-05, 0.20801, 0.00103, 0.11083, 0.00036, 0.00536, 1e-05, 0.44856, 1e-05, 0.04263, 1e-05, 0.00904, 0.0, 0.00344, 0.0, 0.01918, 0.0, 0.04918, 1e-05, 0.39464, 0.00052, 0.05988, 2e-05, 0.47679, 1e-05, 0.1749, 1e-05, 0.0979, 3e-05, 0.01298, 1e-05, 0.36219, 0.0, 0.06366, 9e-05, 0.33335, 0.00033, 0.03303, 2e-05, 0.5241, 1e-05, 0.13498, 0.0, 0.17659, 3e-05, 0.02852, 0.0, 0.19348, 0.0, 0.17987, 1e-05, 0.42044, 0.00016, 0.09432, 0.00048, 0.49108, 2e-05, 0.17986, 0.0, 0.41175, 2e-05, 0.06484, 5e-05, 0.51718, 6e-05, 0.15373, 0.00036, 0.17677, 2e-05, 0.03231, 2e-05, 0.39785, 0.0, 0.11785, 0.0, 0.01579, 0.0, 0.02168, 1e-05, 0.05949, 0.0, 0.09017, 0.00011, 0.24468, 0.0, 0.0746, 1e-05, 0.40593, 0.0, 0.10319, 0.0, 0.01588, 0.0, 0.00905, 1e-05, 0.0263, 0.0, 0.03379, 0.00045, 0.00475, 5e-05, 0.00247, 4e-05, 0.00651, 0.0, 0.00318, 0.0, 0.00512, 0.0, 0.00345, 0.0, 0.00709, 0.0, 0.00279, 6e-05, 0.01037, 3e-05, 0.00338, 0.00035, 0.03782, 3e-05, 0.0058, 7e-05, 0.00921, 2e-05, 0.0039, 0.0001, 0.0199, 2e-05, 0.00633, 0.00044, 0.00373, 6e-05, 0.00229, 1e-05, 0.00959, 0.0, 0.00276, 0.0, 0.00253, 0.0, 0.00227, 0.0, 0.00365, 0.0, 0.00271, 4e-05, 0.00646, 0.00035, 0.00272, 0.0001, 0.02805, 0.00035, 0.0051, 0.00028, 0.00295, 0.0001, 0.00278, 5e-05, 0.00958, 1e-05, 0.00371, 0.00011, 0.00548, 7e-05, 0.00263, 8e-05, 0.00483, 0.0, 0.00305, 0.0, 0.00376, 0.0, 0.00242, 1e-05, 0.00239, 1e-05, 0.00167, 0.00036, 0.00705, 0.00024, 0.00291, 0.00028, 0.02603, 0.00027, 0.00578, 3e-05, 0.00481, 6e-05, 0.00308, 0.00017, 0.00893, 6e-05, 0.00347, 0.0002, 0.22885, 1e-05, 0.06253, 1e-05, 0.38328, 0.0, 0.06729, 0.0, 0.01316, 0.0, 0.00562, 1e-05, 0.01134, 0.0, 0.01409, 7e-05, 0.00671, 5e-05, 0.0034, 0.00013, 0.01916, 7e-05, 0.00269, 0.0, 0.00174, 5e-05, 0.00582, 1e-05, 0.00526, 2e-05, 0.00613, 2e-05, 0.00219, 0.00127, 0.0061, 0.00149, 0.00535, 0.0001, 0.00405, 0.00026, 0.00306, 0.00016, 0.00799, 0.00011, 0.00618, 8e-05, 0.00409, 3e-05, 0.00445, 0.00112, 0.01212, 0.00159, 0.01179, 0.00013, 0.00852, 0.00051, 0.00432, 0.00032, 0.00941, 0.00043, 0.01089, 0.00025, 0.00825, 0.00035, 0.00151, 0.00075, 0.00331, 0.00115, 0.00362, 9e-05, 0.00211, 0.00091, 0.00104, 7e-05, 0.00115, 1e-05, 0.00071, 1e-05, 0.00058, 0.00101, 0.00321, 0.00154, 0.00713, 0.0003, 0.01026, 9e-05, 0.00551, 6e-05, 0.00113, 0.00016, 0.00098, 0.00017, 0.00302, 4e-05, 0.00126, 0.00023, 0.00304, 0.00097, 0.00505, 0.00069, 0.00413, 4e-05, 0.00319, 0.00085, 0.00327, 0.00021, 0.00454, 2e-05, 0.00309, 5e-05, 0.00208, 0.0003, 0.00432, 0.00039, 0.00869, 0.00154, 0.00801, 0.00028, 0.00513, 7e-05, 0.00436, 8e-05, 0.00643, 0.00017, 0.00759, 0.00011, 0.00398, 0.00017, 0.00136, 0.00032, 0.00151, 0.00037, 0.00307, 1e-05, 0.00116, 3e-05, 0.00048, 0.0, 0.00046, 1e-05, 0.00023, 0.0, 0.00019, 0.0001, 0.00203, 0.00019, 0.00247, 0.00017, 0.00373, 0.0, 0.00137, 8e-05, 0.00048, 0.0, 0.00057, 2e-05, 0.00025, 0.0, 0.00024, 0.0003, 0.00571, 0.0012, 0.01467, 0.00229, 0.01571, 0.0001, 0.01071, 0.00097, 0.00585, 0.00028, 0.01213, 0.00012, 0.01405, 5e-05, 0.00953, 0.00013, 0.00857, 0.00028, 0.02261, 0.00175, 0.02305, 0.00012, 0.01649, 0.00085, 0.007, 7e-05, 0.01567, 0.00084, 0.0175, 6e-05, 0.01408, 0.00065, 0.00342, 0.00044, 0.00773, 0.00187, 0.01492, 2e-05, 0.00658, 0.00031, 0.00201, 4e-05, 0.00183, 3e-05, 0.00281, 0.0, 0.00176, 0.00039, 0.00493, 0.00074, 0.01191, 0.00168, 0.01644, 0.00012, 0.00901, 0.00096, 0.00176, 8e-05, 0.0019, 0.00011, 0.00547, 1e-05, 0.00184, 0.00113, 0.00896, 0.00088, 0.01321, 0.00092, 0.01368, 5e-05, 0.00874, 0.00021, 0.00708, 3e-05, 0.00724, 4e-05, 0.00756, 1e-05, 0.00428, 3e-05, 0.0153, 0.00024, 0.02293, 0.00043, 0.01929, 4e-05, 0.01322, 0.00013, 0.01215, 2e-05, 0.01247, 0.00014, 0.01198, 1e-05, 0.00748, 0.00018, 0.00205, 0.00024, 0.00261, 0.00015, 0.00364, 0.0, 0.00138, 1e-05, 0.00048, 1e-05, 0.00054, 1e-05, 0.00028, 0.0, 0.00021, 3e-05, 0.00833, 0.00013, 0.00577, 0.0001, 0.00742, 1e-05, 0.00257, 9e-05, 0.00156, 3e-05, 0.00055, 0.0, 0.00096, 1e-05, 0.00037, 0.0, 0.00399, 0.00063, 0.00719, 0.00043, 0.00977, 1e-05, 0.00525, 0.00023, 0.00348, 0.00013, 0.00652, 1e-05, 0.01495, 1e-05, 0.00392, 1e-05, 0.00853, 0.00034, 0.01458, 0.00058, 0.01978, 4e-05, 0.0113, 0.00028, 0.0068, 6e-05, 0.00905, 8e-05, 0.0161, 4e-05, 0.0082, 0.00068, 0.00234, 0.00074, 0.00262, 7e-05, 0.00645, 0.0, 0.00239, 1e-05, 0.00145, 2e-05, 0.00119, 0.0, 0.00229, 0.0, 0.00086, 2e-05, 0.00363, 0.00062, 0.00601, 2e-05, 0.0132, 0.00013, 0.00546, 2e-05, 0.00132, 0.0001, 0.00108, 7e-05, 0.00446, 2e-05, 0.00098, 5e-05, 0.03803, 0.00031, 0.01711, 2e-05, 0.01126, 0.0, 0.00788, 1e-05, 0.0206, 2e-05, 0.00652, 0.0, 0.01285, 0.0, 0.0049, 4e-05, 0.02777, 0.00013, 0.01992, 0.00032, 0.01534, 2e-05, 0.01167, 1e-05, 0.05077, 2e-05, 0.01859, 9e-05, 0.01315, 0.0001, 0.00714, 0.00014, 0.01218, 0.00022, 0.0063, 7e-05, 0.00358, 0.0, 0.00208, 4e-05, 0.00051, 0.0, 0.00052, 0.0, 0.00017, 0.0, 0.00041, 2e-05, 0.02853, 0.00013, 0.01072, 0.0001, 0.00775, 0.0, 0.00284, 9e-05, 0.00102, 0.0, 0.00063, 1e-05, 0.00018, 0.0, 0.00028, 1e-05, 0.01242, 0.00074, 0.02267, 0.00044, 0.02914, 2e-05, 0.01499, 4e-05, 0.00738, 0.00029, 0.00922, 2e-05, 0.03131, 4e-05, 0.01158, 6e-05, 0.03137, 0.00022, 0.04613, 0.00025, 0.06197, 0.00043, 0.03757, 0.00101, 0.01849, 0.00012, 0.02511, 0.00021, 0.04495, 0.00047, 0.02573, 0.00156, 0.0038, 0.00084, 0.00603, 0.00028, 0.02039, 1e-05, 0.00799, 1e-05, 0.00223, 5e-05, 0.00187, 1e-05, 0.00641, 0.0, 0.00348, 0.00011, 0.00585, 0.00047, 0.01668, 4e-05, 0.02609, 0.001, 0.0147, 0.0015, 0.00158, 0.00013, 0.00249, 0.00012, 0.01481, 4e-05, 0.00293, 0.00015, 0.00779, 0.00054, 0.00963, 0.00012, 0.00444, 1e-05, 0.00497, 2e-05, 0.00338, 3e-05, 0.00427, 3e-05, 0.00241, 1e-05, 0.00391, 1e-05, 0.00985, 0.00011, 0.0172, 0.0002, 0.01379, 0.00027, 0.0166, 0.00118, 0.00508, 1e-05, 0.00942, 7e-05, 0.00574, 0.00019, 0.00963, 0.00042, 0.02931, 0.00024, 0.01076, 0.00011, 0.00848, 0.0, 0.00322, 3e-05, 0.00116, 0.0, 0.00073, 0.0, 0.00021, 0.0, 0.00025, 2e-05, 0.00514, 0.00015, 0.00392, 3e-05, 0.00397, 1e-05, 0.00223, 0.00102, 0.01018, 2e-05, 0.00224, 3e-05, 0.0017, 0.0, 0.00161, 0.0001, 0.00145, 2e-05, 0.002, 0.00013, 0.00265, 0.0, 0.00187, 0.0, 0.00146, 0.0, 0.00167, 0.0, 0.00141, 0.0, 0.00142, 1e-05, 0.00185, 2e-05, 0.00219, 0.00012, 0.00329, 0.0, 0.00201, 0.0, 0.00162, 0.0, 0.00193, 4e-05, 0.00248, 1e-05, 0.00169, 7e-05, 0.00136, 2e-05, 0.00193, 5e-05, 0.00246, 0.0, 0.00151, 0.0, 0.00075, 0.0, 0.00063, 0.0, 0.00045, 0.0, 0.00054, 2e-05, 0.0024, 4e-05, 0.00257, 0.0, 0.00429, 0.0, 0.00177, 0.0, 0.00072, 1e-05, 0.00063, 1e-05, 0.00164, 0.0, 0.00064, 3e-05, 0.00176, 3e-05, 0.00184, 4e-05, 0.00284, 0.0, 0.00187, 0.0, 0.00151, 0.0, 0.00147, 0.0, 0.00114, 0.0, 0.00134, 2e-05, 0.00204, 2e-05, 0.00202, 0.00016, 0.00346, 1e-05, 0.00196, 0.0, 0.00195, 0.0, 0.00185, 7e-05, 0.00264, 1e-05, 0.0017, 3e-05, 0.00131, 5e-05, 0.00104, 6e-05, 0.00195, 0.0, 0.00096, 0.0, 0.0003, 0.0, 0.00033, 1e-05, 0.0002, 0.0, 0.00029, 1e-05, 0.00185, 3e-05, 0.00138, 5e-05, 0.0029, 0.0, 0.0011, 1e-05, 0.00035, 0.0, 0.00036, 1e-05, 0.00022, 0.0, 0.00028, 0.0, 0.00254, 1e-05, 0.00334, 0.00015, 0.00559, 0.0, 0.00316, 0.0, 0.00223, 0.0, 0.00243, 1e-05, 0.00277, 0.0, 0.00265, 2e-05, 0.00369, 0.0, 0.00374, 0.00028, 0.00731, 0.0, 0.00325, 2e-05, 0.00297, 0.0, 0.00329, 9e-05, 0.0053, 1e-05, 0.00345, 0.00015, 0.00198, 1e-05, 0.00276, 0.00019, 0.00563, 0.0, 0.00265, 0.0, 0.00089, 0.0, 0.00082, 0.0, 0.00102, 0.0, 0.00122, 1e-05, 0.00352, 2e-05, 0.00336, 9e-05, 0.00765, 0.0, 0.00265, 3e-05, 0.00058, 0.0, 0.00088, 1e-05, 0.00207, 0.0, 0.00108, 0.00012, 0.00384, 3e-05, 0.00317, 0.00018, 0.00634, 0.0, 0.00297, 1e-05, 0.00276, 0.0, 0.00212, 3e-05, 0.00219, 0.0, 0.00212, 0.0, 0.00443, 1e-05, 0.00365, 0.00013, 0.00739, 0.0, 0.00305, 1e-05, 0.00341, 0.0, 0.00296, 5e-05, 0.00475, 0.0, 0.00333, 3e-05, 0.00192, 5e-05, 0.00147, 8e-05, 0.00295, 0.0, 0.00119, 1e-05, 0.00037, 0.0, 0.0004, 0.0, 0.00024, 0.0, 0.00032, 0.0, 0.00356, 1e-05, 0.00173, 8e-05, 0.00425, 0.0, 0.00128, 2e-05, 0.00065, 1e-05, 0.0007, 0.0, 0.00039, 0.0, 0.00042, 0.0, 0.00261, 0.0002, 0.00362, 8e-05, 0.00387, 0.0, 0.00338, 0.0, 0.00135, 0.0, 0.00235, 0.0, 0.00261, 0.0, 0.00229, 0.0, 0.00364, 4e-05, 0.00359, 0.00011, 0.00557, 2e-05, 0.0029, 0.00013, 0.00223, 1e-05, 0.00308, 1e-05, 0.00323, 2e-05, 0.00226, 0.00061, 0.00132, 0.00031, 0.00183, 2e-05, 0.00154, 0.0, 0.0013, 1e-05, 0.00056, 0.0, 0.0007, 0.0, 0.00041, 0.0, 0.00046, 0.0, 0.00269, 0.00027, 0.00324, 1e-05, 0.00514, 1e-05, 0.00205, 1e-05, 0.0004, 1e-05, 0.00069, 3e-05, 0.00173, 0.0, 0.00049, 1e-05, 0.00188, 9e-05, 0.00188, 3e-05, 0.00244, 0.0, 0.00263, 1e-05, 0.00108, 1e-05, 0.00137, 0.0, 0.00082, 0.0, 0.00219, 1e-05, 0.00183, 4e-05, 0.00162, 6e-05, 0.00338, 2e-05, 0.00207, 2e-05, 0.00133, 1e-05, 0.00137, 1e-05, 0.00139, 4e-05, 0.00162, 0.00012, 0.00081, 2e-05, 0.00096, 8e-05, 0.00103, 0.0, 0.00087, 1e-05, 0.00016, 0.0, 0.0004, 0.0, 7e-05, 0.0, 0.00032, 3e-05, 0.00163, 2e-05, 0.00109, 0.00013, 0.00226, 0.0, 0.00112, 2e-05, 0.0001, 0.0, 0.00038, 1e-05, 5e-05, 0.0, 0.00018, 3e-05, 0.00185, 7e-05, 0.00387, 0.0001, 0.00465, 0.0, 0.00729, 1e-05, 0.00121, 0.0, 0.00215, 1e-05, 0.00275, 0.0, 0.00367, 7e-05, 0.00232, 3e-05, 0.00506, 0.00044, 0.00739, 0.0001, 0.01167, 0.00014, 0.00119, 2e-05, 0.00281, 7e-05, 0.005, 8e-05, 0.01126, 0.0008, 0.00103, 4e-05, 0.00184, 4e-05, 0.00825, 0.0, 0.00587, 2e-05, 0.00096, 0.0, 0.00211, 1e-05, 0.0043, 0.0, 0.00481, 0.0, 0.00074, 6e-05, 0.00168, 4e-05, 0.00424, 6e-05, 0.00468, 0.00236, 0.00067, 1e-05, 0.00158, 6e-05, 0.00413, 1e-05, 0.00133, 0.00012, 0.00219, 5e-05, 0.00193, 0.00012, 0.00271, 0.0, 0.00269, 3e-05, 0.00148, 1e-05, 0.00148, 3e-05, 0.00184, 0.0, 0.00174, 1e-05, 0.00223, 4e-05, 0.0035, 0.0004, 0.00357, 3e-05, 0.00489, 0.00034, 0.0015, 4e-05, 0.00198, 7e-05, 0.00112, 4e-05, 0.00271, 0.00042, 0.00164, 4e-05, 0.00111, 0.00019, 0.00249, 0.0, 0.00128, 1e-05, 0.0001, 0.0, 0.00041, 1e-05, 6e-05, 0.0, 0.00022, 8e-05, 0.00229, 7e-05, 0.00158, 0.00035, 0.00333, 8e-05, 0.00291, 0.00028, 0.00247, 1e-05, 0.00174, 1e-05, 0.00039, 0.0, 0.00103, 1e-05, 0.00063, 0.00394, 0.00077, 0.00705, 0.00095, 0.0001, 0.00065, 0.00323, 0.00091, 0.00021, 0.00157, 0.001, 0.00075, 0.00016, 0.00064, 2e-05, 0.00062, 0.00288, 0.00074, 0.00447, 0.0008, 0.00058, 0.00063, 0.0011, 0.00067, 0.00065, 0.00077, 0.00136, 0.00077, 0.00033, 0.00064, 0.00167, 0.00079, 0.00096, 0.00101, 0.00365, 0.0011, 8e-05, 0.00066, 0.00478, 0.00055, 6e-05, 0.00098, 8e-05, 0.00035, 5e-05, 0.00038, 0.00167, 0.00071, 0.00447, 0.00094, 0.00182, 0.0013, 0.00027, 0.00099, 0.00064, 0.00041, 0.00052, 0.00046, 0.0031, 0.00068, 0.00027, 0.00068, 0.00083, 0.0005, 0.00179, 0.00058, 0.00347, 0.00072, 3e-05, 0.00046, 0.00166, 0.00075, 0.00013, 0.00137, 0.00185, 0.00049, 0.00014, 0.00042, 0.0008, 0.00051, 0.00223, 0.0006, 0.00617, 0.00063, 0.00028, 0.00046, 0.00052, 0.00052, 0.00036, 0.0006, 0.00381, 0.00068, 0.00043, 0.00051, 0.00116, 0.0004, 0.0007, 0.00055, 0.02093, 0.0007, 5e-05, 0.00046, 0.00475, 0.00049, 0.00087, 0.00061, 0.00269, 0.0001, 0.00753, 0.00015, 0.00868, 0.00036, 0.00088, 0.00046, 0.01093, 0.00061, 0.00065, 0.00036, 0.01209, 0.0002, 0.0003, 0.0005, 0.00312, 7e-05, 0.01122, 0.00012, 0.01057, 0.00069, 0.00128, 0.00094, 0.00502, 0.00109, 3e-05, 0.00084, 0.0102, 0.00099, 0.00022, 0.00222, 0.00121, 0.00093, 0.00016, 0.001, 6e-05, 0.00065, 0.00075, 0.00088, 0.00357, 0.00089, 0.00084, 0.00076, 0.00332, 0.00071, 0.00025, 0.00092, 0.0049, 0.00084, 0.00064, 0.00072, 0.00352, 0.00099, 0.00062, 0.00147, 0.00422, 0.00165, 2e-05, 0.00105, 0.01208, 0.00087, 0.00017, 0.00184, 0.00074, 0.00085, 8e-05, 0.00084, 0.00067, 0.00075, 0.00301, 0.00118, 0.02122, 0.00142, 0.00095, 0.00119, 0.00354, 0.00041, 0.00066, 0.00072, 0.00092, 0.00084, 0.00014, 0.00081, 0.01508, 0.00051, 0.00101, 0.0006, 0.00151, 0.00078, 2e-05, 0.00054, 0.0061, 0.00071, 0.00013, 0.00146, 0.00909, 0.00049, 0.00024, 0.0005, 0.00208, 0.0005, 0.00226, 0.00062, 0.00275, 0.00063, 0.00155, 0.00051, 0.00032, 0.0005, 0.00036, 0.00063, 0.00629, 0.00061, 0.00057, 0.00053, 0.00078, 0.00037, 0.00125, 0.00046, 0.02389, 0.0006, 5e-05, 0.00032, 0.00803, 0.00021, 0.00011, 0.00047, 2e-05, 6e-05, 0.00925, 9e-05, 0.00982, 0.00041, 0.00214, 0.0005, 0.0043, 0.00055, 0.00113, 0.00038, 0.0001, 0.00021, 0.00029, 0.0003, 0.00055, 0.00012, 0.00105, 0.0001, 2e-05, 0.00113, 0.00056, 0.00101, 0.00224, 0.00173, 1e-05, 0.00092, 0.00302, 0.00173, 2e-05, 0.00198, 0.00016, 0.00117, 2e-05, 0.00078, 2e-05, 0.00119, 0.00032, 0.00098, 0.00216, 0.00154, 8e-05, 0.00079, 0.00082, 0.00131, 3e-05, 0.00107, 0.00043, 0.00146, 8e-05, 0.00076, 0.00253, 0.00146, 0.00136, 0.00137, 0.00046, 0.00209, 1e-05, 0.001, 0.00402, 0.00077, 3e-05, 0.00116, 0.00056, 0.00048, 6e-05, 0.00051, 0.00121, 0.00122, 0.0009, 0.00128, 0.00095, 0.00241, 6e-05, 0.0014, 0.00038, 0.00078, 0.00069, 0.00071, 0.00129, 0.00127, 0.0001, 0.00096, 0.00011, 0.00075, 0.00051, 0.00076, 0.00022, 0.00102, 0.0, 0.00059, 0.00302, 0.00115, 8e-05, 0.00164, 0.00094, 0.00058, 8e-05, 0.00045, 0.00091, 0.00095, 0.00022, 0.00085, 0.0028, 0.00107, 9e-05, 0.0006, 0.00072, 0.00104, 0.00018, 0.00087, 0.00304, 0.00111, 0.00103, 0.00061, 0.00298, 0.00074, 0.00058, 0.00087, 0.00211, 0.00109, 3e-05, 0.0007, 0.00263, 0.00089, 5e-05, 0.00093, 1e-05, 0.00014, 0.00038, 0.00021, 0.00167, 0.00088, 0.00028, 0.00089, 0.00071, 0.00109, 0.00078, 0.00058, 0.0124, 0.00054, 1e-05, 0.00089, 2e-05, 0.00011, 0.0002, 0.00019, 0.00281, 0.00093, 0.00086, 0.00113, 0.00068, 0.00151, 1e-05, 0.00084, 0.00676, 0.00137, 7e-05, 0.00231, 0.0001, 0.00126, 1e-05, 0.00099, 0.00021, 0.00109, 0.00058, 0.00117, 0.00204, 0.00144, 0.00022, 0.00081, 0.01398, 0.00107, 0.00024, 0.00115, 0.00071, 0.00126, 0.0001, 0.00076, 0.0074, 0.00112, 0.00257, 0.0016, 0.00055, 0.00257, 6e-05, 0.00119, 0.0122, 0.00056, 0.00107, 0.00124, 0.00037, 0.00068, 0.00078, 0.0008, 0.00114, 0.00105, 0.00218, 0.00158, 0.00128, 0.00222, 0.00052, 0.00144, 0.00734, 0.0002, 0.00103, 0.00081, 0.00019, 0.00087, 0.00031, 0.00096, 0.00057, 0.0005, 0.00126, 0.00058, 0.00874, 0.00069, 3e-05, 0.00037, 0.00974, 0.00044, 0.00127, 0.00091, 0.00423, 0.00035, 0.0036, 0.00026, 0.00261, 0.00069, 0.00086, 0.00079, 0.00299, 0.00071, 0.00063, 0.00046, 0.04573, 0.00046, 0.00134, 0.00058, 0.00384, 0.00054, 0.00343, 0.00045, 0.00453, 0.00088, 0.00025, 0.0009, 0.00071, 0.00105, 2e-05, 0.00052, 0.00341, 0.00054, 4e-05, 0.00085, 0.0, 9e-05, 0.0, 0.00012, 0.0004, 0.00074, 0.00178, 0.00071, 0.00157, 0.0006, 0.00124, 0.00028, 0.01022, 0.00033, 0.0002, 0.00063, 0.00038, 0.00012, 0.00014, 0.0002, 0.00019, 0.00108, 0.0001, 0.0005, 0.00028, 0.0016, 1e-05, 0.00049, 0.00062, 0.0014, 0.0, 0.00087, 9e-05, 0.00075, 5e-05, 0.00047, 3e-05, 0.00112, 1e-05, 0.00052, 0.00058, 0.00188, 2e-05, 0.00053, 6e-05, 0.00115, 1e-05, 0.00062, 0.00057, 0.00168, 0.00015, 0.00063, 0.00041, 0.00097, 0.00028, 0.00048, 0.00039, 0.00128, 1e-05, 0.0004, 0.00043, 0.00099, 2e-05, 0.0007, 0.00021, 0.00042, 6e-05, 0.00027, 0.00064, 0.00131, 0.00021, 0.00059, 0.00022, 0.00216, 6e-05, 0.00053, 0.0001, 0.00073, 4e-05, 0.00037, 0.00043, 0.00099, 6e-05, 0.00041, 0.00104, 0.00073, 0.00034, 0.00039, 0.00082, 0.00117, 3e-05, 0.0004, 0.00024, 0.00104, 9e-05, 0.00082, 0.00045, 0.00079, 0.00031, 0.0004, 0.00091, 0.00086, 4e-05, 0.00042, 0.00034, 0.0016, 0.00029, 0.00044, 2e-05, 0.00094, 0.0, 0.0005, 8e-05, 0.00177, 0.00069, 0.00057, 0.00056, 0.00051, 1e-05, 0.0003, 1e-05, 0.00073, 5e-05, 0.00029, 0.00029, 0.00041, 0.0, 0.00045, 1e-05, 0.0001, 0.00029, 0.00013, 0.00063, 0.00054, 0.0, 0.00029, 1e-05, 0.00102, 9e-05, 0.00031, 0.00083, 0.00042, 0.0, 0.00042, 2e-05, 7e-05, 1e-05, 0.00011, 0.00072, 0.00158, 0.00011, 0.00072, 0.00094, 0.003, 3e-05, 0.00062, 0.00176, 0.00167, 8e-05, 0.00098, 0.0009, 0.00082, 0.00035, 0.00048, 0.00019, 0.00374, 0.00036, 0.00076, 0.0012, 0.00417, 0.00081, 0.00079, 0.00115, 0.00345, 7e-05, 0.0006, 0.0008, 0.00423, 0.00011, 0.00061, 0.0008, 0.00081, 0.00043, 0.00071, 0.00012, 0.0014, 2e-05, 0.00057, 0.0006, 0.00041, 0.0, 0.00047, 2e-05, 0.00032, 5e-05, 0.00038, 0.00047, 0.00327, 0.0026, 0.00072, 0.00029, 0.00544, 0.00039, 0.00065, 0.01764, 0.00023, 1e-05, 0.0002, 4e-05, 0.00085, 3e-05, 0.00036, 0.00246, 0.00112, 0.00013, 0.00067, 0.00023, 0.00297, 0.00015, 0.00054, 0.00016, 0.00118, 0.0, 0.00097, 1e-05, 0.00079, 0.00047, 0.00048, 0.00051, 0.00296, 0.00047, 0.00079, 0.00012, 0.0031, 0.0003, 0.00074, 0.0002, 0.00211, 0.0, 0.00064, 9e-05, 0.00285, 7e-05, 0.00074, 0.00017, 0.00052, 1e-05, 0.00029, 1e-05, 0.00096, 0.0, 0.00028, 3e-05, 0.00042, 0.0, 0.00044, 1e-05, 7e-05, 0.0, 9e-05, 0.00026, 0.00178, 1e-05, 0.00047, 1e-05, 0.003, 0.0, 0.00038, 3e-05, 0.00061, 1e-05, 0.00048, 4e-05, 0.00018, 0.0, 0.00018, 1e-05, 0.00045, 0.00133, 0.0002, 0.00141, 0.00071, 2e-05, 0.00034, 0.00129, 0.00053, 0.00011, 0.00029, 0.00015, 0.00055, 0.00013, 0.00024, 2e-05, 0.00229, 0.00049, 0.00085, 0.00061, 0.00258, 3e-05, 0.00092, 0.0003, 0.00271, 8e-05, 0.00082, 0.0001, 0.00256, 9e-05, 0.00092, 0.00215, 0.00054, 0.00145, 0.00028, 2e-05, 0.00096, 7e-05, 0.0004, 0.00065, 0.00019, 4e-05, 0.00023, 1e-05, 0.0002, 8e-05, 0.0002, 0.00014, 0.00242, 0.00033, 0.00091, 2e-05, 0.00431, 0.0, 0.00093, 7e-05, 0.00067, 7e-05, 0.0003, 3e-05, 0.00142, 0.0, 0.00053, 2e-05, 0.00025, 0.001, 0.00046, 2e-05, 0.00057, 7e-05, 0.00053, 0.00022, 0.00036, 6e-05, 0.00077, 0.0, 0.00044, 0.00019, 0.00044, 2e-05, 0.0013, 0.00017, 0.0011, 8e-05, 0.00163, 5e-05, 0.00093, 1e-05, 0.00071, 6e-05, 0.00092, 1e-05, 0.00127, 2e-05, 0.00125, 0.0001, 0.00026, 0.00015, 0.00093, 3e-05, 0.00126, 0.0, 0.00078, 2e-05, 0.00055, 0.0, 0.0006, 1e-05, 0.00017, 0.0, 0.00041, 0.00018, 0.0005, 3e-05, 0.00104, 1e-05, 0.0013, 0.0, 0.00077, 1e-05, 0.00053, 0.0, 0.00053, 1e-05, 5e-05, 0.0, 0.00038, 0.0005, 5e-05, 0.00054, 4e-05, 0.00026, 0.0001, 4e-05, 4e-05, 0.00096, 0.0001, 3e-05, 0.00012, 1e-05, 0.0001, 9e-05, 6e-05, 0.00017, 7e-05, 0.00014, 4e-05, 0.00029, 0.00018, 4e-05, 4e-05, 0.00055, 6e-05, 4e-05, 5e-05, 8e-05, 0.00014, 8e-05, 5e-05, 0.00053, 7e-05, 0.0002, 6e-05, 2e-05, 0.00014, 0.0, 7e-05, 0.00013, 7e-05, 0.0, 0.00011, 1e-05, 6e-05, 0.0, 7e-05, 5e-05, 7e-05, 0.00082, 5e-05, 0.00011, 0.00019, 1e-05, 7e-05, 0.003, 3e-05, 0.0001, 6e-05, 4e-05, 0.0001, 1e-05, 0.00011, 0.00011, 0.00026, 0.00015, 0.00025, 1e-05, 0.0004, 0.0, 0.0004, 1e-05, 0.00021, 0.0, 0.00048, 0.0, 0.00026, 0.0, 0.00027, 5e-05, 0.00029, 2e-05, 0.00023, 4e-05, 0.0006, 2e-05, 0.00031, 7e-05, 0.00018, 1e-05, 0.00034, 4e-05, 0.00025, 4e-05, 0.00059, 3e-05, 0.00053, 6e-05, 0.001, 2e-05, 0.00125, 0.0, 0.00074, 1e-05, 0.00052, 0.0, 0.00041, 0.0, 4e-05, 0.0, 0.00035, 0.00022, 0.00055, 2e-05, 0.00078, 3e-05, 0.00055, 1e-05, 0.00045, 4e-05, 0.00045, 1e-05, 0.00132, 2e-05, 0.00016, 1e-05, 0.00051, 0.00017, 0.00382, 0.00083, 0.0101, 0.00187, 0.00395, 4e-05, 0.00389, 0.00039, 0.00399, 0.00028, 0.00713, 0.00031, 0.00353, 5e-05, 0.0033, 2e-05, 0.00424, 0.00067, 0.01268, 0.00129, 0.00473, 0.00031, 0.00611, 0.00123, 0.00376, 0.00033, 0.0101, 0.00045, 0.0044, 0.00023, 0.00647, 0.00043, 0.00279, 0.00032, 0.00582, 0.00024, 0.00308, 3e-05, 0.00237, 0.00069, 0.00136, 2e-05, 0.00099, 0.0, 0.00097, 0.0, 0.00101, 0.00012, 0.00289, 0.00093, 0.00686, 0.00047, 0.00387, 0.00016, 0.00343, 9e-05, 0.00198, 0.0001, 0.00248, 0.00021, 0.00254, 1e-05, 0.00197, 6e-05, 0.00309, 0.00079, 0.00423, 0.00017, 0.00331, 2e-05, 0.00249, 0.00046, 0.0031, 9e-05, 0.00252, 1e-05, 0.00307, 1e-05, 0.00208, 8e-05, 0.00407, 0.00035, 0.00732, 0.00066, 0.00403, 0.00039, 0.00361, 0.00029, 0.00352, 0.00016, 0.0058, 3e-05, 0.00403, 8e-05, 0.00398, 8e-05, 0.00176, 0.00013, 0.002, 3e-05, 0.00256, 1e-05, 0.00144, 0.0001, 0.00038, 1e-05, 0.00029, 1e-05, 0.00049, 0.0, 0.00048, 0.00052, 0.00234, 0.00013, 0.00245, 3e-05, 0.00304, 1e-05, 0.00167, 0.00023, 0.00063, 0.0, 0.00034, 1e-05, 0.00057, 1e-05, 0.00049, 0.00045, 0.00509, 0.00047, 0.01325, 0.00117, 0.00534, 1e-05, 0.00573, 0.00118, 0.00507, 0.00011, 0.00563, 0.00014, 0.00453, 2e-05, 0.00451, 3e-05, 0.00546, 0.0002, 0.01485, 0.00055, 0.00551, 0.00017, 0.00623, 0.00198, 0.00484, 8e-05, 0.01229, 0.00055, 0.00503, 0.00012, 0.00706, 0.00079, 0.00341, 0.00018, 0.00702, 0.00041, 0.00455, 0.0, 0.00331, 0.00061, 0.00153, 3e-05, 0.00078, 1e-05, 0.00123, 0.0, 0.00112, 6e-05, 0.00334, 0.00048, 0.00797, 0.00174, 0.00394, 0.0001, 0.00373, 0.00102, 0.00174, 5e-05, 0.00246, 4e-05, 0.00248, 1e-05, 0.00169, 0.00196, 0.00512, 0.00048, 0.00853, 0.00032, 0.00567, 1e-05, 0.00449, 0.00079, 0.00504, 6e-05, 0.00246, 9e-05, 0.00454, 1e-05, 0.00302, 1e-05, 0.00591, 0.00066, 0.01162, 0.00065, 0.00536, 0.00029, 0.0048, 0.00022, 0.0052, 0.00014, 0.00823, 0.00024, 0.00491, 5e-05, 0.00494, 0.00014, 0.00234, 9e-05, 0.00233, 0.00026, 0.00305, 1e-05, 0.00155, 0.00019, 0.00065, 2e-05, 0.00036, 1e-05, 0.00056, 0.0, 0.00049, 0.0003, 0.00375, 0.00013, 0.00335, 0.00024, 0.00367, 7e-05, 0.00172, 3e-05, 0.00085, 1e-05, 0.00035, 1e-05, 0.00055, 0.0, 0.00039, 0.0, 0.00674, 0.00044, 0.00931, 0.0003, 0.00542, 0.0, 0.0048, 0.00026, 0.00392, 4e-05, 0.00493, 1e-05, 0.0029, 0.0, 0.0023, 2e-05, 0.00667, 0.00013, 0.01188, 0.00046, 0.00573, 4e-05, 0.00689, 0.00061, 0.00447, 3e-05, 0.0073, 7e-05, 0.00375, 4e-05, 0.00521, 0.00057, 0.00409, 0.0002, 0.00419, 9e-05, 0.00358, 0.0, 0.00228, 0.00037, 0.00102, 1e-05, 0.00097, 1e-05, 0.00129, 0.0, 0.00098, 0.00016, 0.00379, 0.00043, 0.00566, 0.00013, 0.00384, 1e-05, 0.00326, 4e-05, 0.00173, 6e-05, 0.00182, 0.00015, 0.00158, 0.0, 0.00139, 1e-05, 0.00263, 0.0002, 0.00191, 1e-05, 0.00418, 0.0, 0.00254, 6e-05, 0.00208, 2e-05, 0.00159, 0.0, 0.0023, 0.0, 0.00179, 6e-05, 0.00427, 7e-05, 0.00398, 4e-05, 0.00538, 1e-05, 0.00528, 7e-05, 0.00294, 5e-05, 0.00263, 4e-05, 0.00364, 4e-05, 0.00457, 0.00018, 0.00137, 0.00011, 0.00107, 0.00015, 0.00184, 0.0, 0.00147, 0.0001, 0.00043, 1e-05, 0.00039, 2e-05, 0.00028, 0.0, 0.00044, 0.00014, 0.00186, 0.00011, 0.00087, 0.0001, 0.00294, 0.0, 0.0019, 0.00061, 0.00064, 0.0, 0.00045, 2e-05, 0.00029, 0.0, 0.00037, 0.00019, 0.00401, 0.00039, 0.01155, 0.00016, 0.00389, 0.0, 0.00468, 0.00163, 0.00264, 0.0001, 0.00387, 2e-05, 0.00194, 0.0, 0.00261, 0.00025, 0.00604, 0.00018, 0.01696, 0.00033, 0.00531, 0.00011, 0.00697, 0.00516, 0.00354, 4e-05, 0.00915, 0.0001, 0.00293, 4e-05, 0.00571, 0.00083, 0.00208, 0.00031, 0.00325, 6e-05, 0.0026, 0.0, 0.00171, 0.00073, 0.00063, 3e-05, 0.00076, 7e-05, 0.00067, 0.0, 0.00084, 0.00164, 0.00296, 0.00047, 0.00501, 0.00013, 0.00299, 4e-05, 0.00258, 0.00051, 0.0004, 0.00016, 0.0013, 8e-05, 0.00078, 1e-05, 0.00091, 6e-05, 0.00248, 0.00034, 0.00167, 8e-05, 0.00261, 0.0, 0.00126, 0.00046, 0.00162, 9e-05, 0.00086, 1e-05, 0.00122, 0.0, 0.00088, 8e-05, 0.00371, 7e-05, 0.00232, 0.00035, 0.00461, 3e-05, 0.002, 0.00073, 0.0027, 6e-05, 0.00153, 8e-05, 0.00244, 0.00025, 0.00186, 0.00019, 0.00187, 0.00014, 0.0008, 0.00012, 0.00293, 0.0, 0.00154, 0.00014, 0.00064, 1e-05, 0.00043, 0.0, 0.00028, 0.0, 0.00034, 0.00011, 0.00209, 0.00029, 0.00044, 0.00026, 0.00145, 2e-05, 0.00039, 0.00221, 0.00095, 8e-05, 0.00033, 5e-05, 0.00045, 6e-05, 0.0003, 6e-05, 0.00073, 2e-05, 0.00062, 2e-05, 0.0008, 0.0, 0.00041, 2e-05, 0.00068, 0.0, 0.00074, 0.0, 0.00059, 0.0, 0.00053, 1e-05, 0.00078, 1e-05, 0.00062, 3e-05, 0.00096, 0.0, 0.00046, 1e-05, 0.00073, 0.0, 0.00067, 1e-05, 0.00082, 1e-05, 0.00055, 4e-05, 0.00083, 5e-05, 0.00065, 0.0, 0.001, 0.0, 0.00033, 9e-05, 0.00056, 0.0, 0.00023, 0.0, 0.00045, 0.0, 0.00032, 0.00018, 0.00107, 3e-05, 0.00081, 1e-05, 0.00119, 0.0, 0.00038, 0.0, 0.00058, 1e-05, 0.00048, 1e-05, 0.00079, 0.0, 0.00038, 0.0, 0.00034, 5e-05, 0.00023, 0.0, 0.0006, 0.0, 0.00021, 7e-05, 0.0005, 1e-05, 0.00038, 0.0, 0.0005, 0.0, 0.00027, 1e-05, 0.00053, 3e-05, 0.00031, 1e-05, 0.00099, 1e-05, 0.0003, 0.0, 0.00037, 1e-05, 0.00028, 1e-05, 0.00098, 1e-05, 0.00045, 0.0, 0.00013, 5e-05, 0.00014, 3e-05, 0.00059, 0.0, 0.00015, 2e-05, 0.00022, 0.0, 8e-05, 0.0, 0.00013, 0.0, 0.00015, 0.00037, 0.00019, 4e-05, 0.00014, 5e-05, 0.00081, 0.0, 0.00015, 2e-05, 0.00022, 0.0, 0.00011, 0.0, 0.00012, 0.0, 0.00014, 0.00031, 0.00083, 1e-05, 0.00067, 1e-05, 0.00096, 0.0, 0.00048, 4e-05, 0.00072, 0.0, 0.00068, 0.0, 0.00063, 0.0, 0.00043, 0.0, 0.00085, 0.0, 0.00062, 2e-05, 0.001, 0.0, 0.00043, 0.0, 0.00079, 0.0, 0.00064, 2e-05, 0.00085, 0.0, 0.00052, 4e-05, 0.00078, 2e-05, 0.00057, 0.0, 0.00098, 0.0, 0.00029, 3e-05, 0.00039, 0.0, 0.0002, 0.0, 0.00031, 0.0, 0.00022, 4e-05, 0.001, 1e-05, 0.00073, 1e-05, 0.00103, 0.0, 0.00036, 3e-05, 0.00024, 0.0, 0.00038, 0.0, 0.00062, 0.0, 0.00041, 0.00011, 0.00058, 4e-05, 0.00039, 0.0, 0.00101, 0.0, 0.00033, 2e-05, 0.00056, 0.0, 0.00042, 1e-05, 0.00053, 0.0, 0.00028, 0.0, 0.00081, 0.0, 0.00043, 1e-05, 0.00125, 0.0, 0.00034, 0.0, 0.00054, 0.0, 0.00039, 1e-05, 0.00099, 1e-05, 0.00052, 1e-05, 0.00019, 9e-05, 0.00014, 3e-05, 0.00075, 0.0, 0.00014, 2e-05, 0.00022, 0.0, 0.00011, 0.0, 0.00012, 0.0, 0.00015, 0.00017, 0.00038, 5e-05, 0.00019, 4e-05, 0.00071, 1e-05, 0.00015, 0.00015, 0.00018, 2e-05, 0.00015, 1e-05, 8e-05, 1e-05, 7e-05, 0.0, 0.00094, 0.00011, 0.00089, 1e-05, 0.00079, 0.0, 0.00092, 1e-05, 0.001, 0.0, 0.00114, 0.0, 0.00071, 0.0, 0.00142, 3e-05, 0.001, 1e-05, 0.00092, 1e-05, 0.00061, 0.0, 0.00078, 1e-05, 0.00099, 0.0, 0.00125, 1e-05, 0.0006, 0.0, 0.00074, 2e-05, 0.00132, 0.00012, 0.00094, 1e-05, 0.0013, 0.0, 0.00115, 4e-05, 0.00028, 0.0, 0.0002, 0.0, 0.00136, 0.0, 0.0003, 0.00014, 0.0013, 9e-05, 0.00098, 1e-05, 0.0007, 0.0, 0.00079, 1e-05, 0.00079, 1e-05, 0.00033, 2e-05, 0.00104, 0.0, 0.00073, 1e-05, 0.00046, 0.00025, 0.00026, 1e-05, 0.00068, 0.0, 0.00034, 2e-05, 0.00044, 1e-05, 0.00036, 0.0, 0.00056, 0.0, 0.00047, 1e-05, 0.00053, 6e-05, 0.0003, 3e-05, 0.00053, 0.0, 0.00045, 0.0, 0.00026, 3e-05, 0.00023, 2e-05, 0.00049, 1e-05, 0.00082, 2e-05, 0.00021, 0.00018, 0.00021, 0.0001, 0.00038, 0.0, 0.00039, 2e-05, 0.00042, 0.0, 0.00016, 2e-05, 0.00031, 0.0, 0.00014, 0.00016, 0.00023, 0.0001, 0.00023, 0.00014, 0.00039, 0.0, 0.00037, 8e-05, 0.00043, 0.0, 0.00025, 4e-05, 0.00027, 0.0, 0.00017, 0.00024, 0.00013, 7e-05, 0.00074, 0.0, 0.00019, 0.0, 0.00061, 6e-05, 0.00015, 0.0, 0.0008, 0.0, 0.00011, 0.0, 0.00062, 0.00016, 0.00015, 1e-05, 0.00117, 3e-05, 0.00015, 0.0, 0.00077, 1e-05, 0.0001, 0.0, 0.001, 1e-05, 0.0001, 0.0, 0.00106, 3e-05, 0.00069, 6e-05, 0.00161, 1e-05, 0.00214, 0.0, 0.00113, 5e-05, 0.00025, 0.0, 0.00039, 0.0, 0.00054, 0.0, 0.00073, 0.00012, 0.0003, 8e-05, 0.00146, 1e-05, 0.00044, 0.0, 0.00077, 0.00014, 0.00013, 1e-05, 0.0003, 1e-05, 0.00056, 0.0, 0.00065, 1e-05, 0.00029, 0.00028, 0.00027, 2e-05, 0.00037, 0.0, 0.00067, 3e-05, 0.00042, 1e-05, 0.00056, 1e-05, 0.00022, 0.0, 0.00058, 0.00025, 0.00033, 4e-05, 0.00034, 2e-05, 0.00047, 0.0, 0.0005, 1e-05, 0.00023, 1e-05, 0.00028, 1e-05, 0.00042, 1e-05, 0.0007, 3e-05, 0.0002, 0.00026, 0.00019, 9e-05, 0.00046, 0.0, 0.00036, 2e-05, 0.00047, 0.0, 0.0002, 1e-05, 0.00037, 0.0, 0.00027, 0.00011, 0.00031, 0.00015, 0.00029, 0.00012, 0.00072, 0.0, 0.00031, 0.00022, 0.00075, 0.0, 0.00054, 3e-05, 0.00057, 0.0, 0.00034, 1e-05])), 0, tol=3e-4)
check("hide 'It': walk", hide[1], 0.00205, tol=5e-5)
check("hide 'went': walk", hide[1 << 10], 0.00219, tol=5e-5)
check("hide 'for': walk", hide[1 << 11], 0.00063, tol=5e-5)
alone = [hide[1 << j] for j in range(T - 1)]
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.bar(range(T - 1), alone, color=INK); ax.axhline(P_WALK, color=INK2, ls="--", lw=1)
ax.set_xticks(range(T - 1), WORDS[:-1], rotation=30); ax.set_ylabel("walk, with this word hidden")
ax.set_title("Hide one word at a time (dashed: nothing hidden)", fontsize=10); plt.show()

### 3. The weather doesn't decide it (Plate III)
Hiding *warm* and *sunny* together still leaves walk well ahead. Changing words, instead of hiding them, says the same.

In [ ]:
print(f"warm and sunny hidden: walk {100 * hide[(1 << 3) | (1 << 5)]:.1f}%")
check("warm + sunny hidden: walk", hide[(1 << 3) | (1 << 5)], 0.38115, tol=2e-4)
for s, expected in {'It was a cold, rainy day, so we went for a': 0.64167, 'It was a warm, sunny day, so we went to a': 0.00113, 'It was a warm, sunny day, so we went out for a': 0.3972, 'We went for a': 0.25125}.items():
    p = torch.softmax(run(encode(s))["logits"][-1], -1)
    top = [decode([j]) for j in p.topk(3).indices.tolist()]
    print(f"{s!r:52} walk {100 * p[WALK]:5.1f}%   top: {top}")
    check(f"walk after “{s}”", p[WALK].item(), expected, tol=2e-4)

## Part 2 · One head, by hand

### 4. Twelve heads in every block (Plate V)
Each block's attention is 12 **heads** side by side. Each head has four weight tables: **W_Q**, **W_K** and **W_V**
(768 × 64 each) turn a word's LayerNormed total into a 64-number *query*, *key* and *value*; **W_O** (64 × 768) turns
what the head gathers back into 768 numbers. In GPT-2's file the three input tables of all twelve heads are stored side
by side in one 768 × 2,304 array, and W_O of all twelve as one 768 × 768 array.

We take **block 11, head 12**, the head that pushes walk second-hardest.

In [ ]:
L_, H_ = 10, 11                                         # block 11, head 12, counted from 0
W = SD[f"h.{L_}.attn.c_attn.weight"]; B = SD[f"h.{L_}.attn.c_attn.bias"]
cols = lambda part: slice(768 * part + 64 * H_, 768 * part + 64 * (H_ + 1))
W_Q, W_K, W_V = (W[:, cols(p)] for p in range(3))
b_Q, b_K, b_V = (B[cols(p)] for p in range(3))
W_O = SD[f"h.{L_}.attn.c_proj.weight"][64 * H_: 64 * (H_ + 1)]
print("W_Q, W_K, W_V:", tuple(W_Q.shape), "  W_O:", tuple(W_O.shape))
fig, ax = plt.subplots(1, 4, figsize=(9, 2.6))
for a, (name, M_) in zip(ax, [("W_Q", W_Q), ("W_K", W_K), ("W_V", W_V), ("W_O (turned)", W_O.T)]):
    a.imshow(M_.abs().reshape(96, 8, 8, 8).mean((1, 3)).cpu(), cmap="gray_r", aspect="auto")
    a.set_title(name, fontsize=10); a.set_xticks([]); a.set_yticks([])
fig.suptitle("Block 11, head 12: its four weight tables (size of each weight, as ink)", fontsize=10, y=1.06); plt.show()

### 5. Three arrows per word (Plate VI)
The head reads each word's total after block 11's first LayerNorm (call it **x**, 768 numbers). It makes, for the word
that is asking (the last *a*), a **query** `q = x W_Q + b_Q`; for every word, a **key** `k = x W_K + b_K` and a **value**
`v = x W_V + b_V`. The query says what the word is looking for, a key what a word offers to be found by, a value what
it would hand over.

In [ ]:
h1 = SD[f"h.{L_}.ln_1.weight"]; s1 = SD[f"h.{L_}.ln_1.bias"]
x = layer_norm(c["resid"][L_], h1, s1)                 # 13 words x 768
q = x[-1] @ W_Q + b_Q                                  # the last word's query
K = x @ W_K + b_K                                      # every word's key   (13 x 64)
V = x @ W_V + b_V                                      # every word's value (13 x 64)
print("the query's first 4 numbers:", np.round(q[:4].tolist(), 4), f"  its length {q.norm():.3f}")
check("the query's first 4 numbers (largest difference)", (q[:4].cpu() - torch.tensor([-0.7652, -1.3172, -1.4509, -0.3349])).abs().max().item(), 0, tol=2e-3)
check("the query's length", q.norm().item(), 11.8208, tol=2e-3)
assert (q - c["q"][L_, H_, -1]).abs().max() < 1e-4                # the same query the model itself made

### 6. A dot product per word (Plate VII)
How well does each word's key match the query? Multiply the two, number by number, and add the 64 products.

In [ ]:
qk = K @ q
for w, v in zip(WORDS, qk.tolist()):
    print(f"  {w!r:9} {v:7.1f}  " + "█" * max(0, round(v / 2)))
check("query · key for 'for'", qk[11].item(), 59.866, tol=0.02)
check("query · key for 'It'", qk[0].item(), 43.606, tol=0.02)
check("every word's query · key (largest difference)", max(abs(a - b) for a, b in zip(qk.tolist(), [43.606, -0.1121, -14.9186, -15.8537, -21.9918, -33.059, -11.9632, -5.3115, -1.902, 6.3901, 19.0856, 59.866, 32.2721])), 0, tol=0.02)

### 7. Divide by 8, then softmax (Plate VIII)
The 13 matches become shares that add up to 1, by Lecture 19's softmax. First they are divided by 8 (the square root of
64, the key's length in numbers), which keeps them from getting too large as keys get longer.

In [ ]:
scores = qk / 8
share = torch.softmax(scores, -1)
for w, s in zip(WORDS, share.tolist()):
    print(f"  {w!r:9} {100 * s:6.2f}%  " + "█" * round(60 * s))
check("share on 'for'", share[11].item(), 0.85341, tol=1e-4)
check("share on 'It'", share[0].item(), 0.1118, tol=1e-4)
check("share on the last 'a'", share[12].item(), 0.02711, tol=1e-4)
assert (share - c["pattern"][L_, H_, -1]).abs().max() < 1e-5        # the same shares the model itself used

### 8. No peeking ahead (Plate IX)
Every word does the same, at the same time, but may only look at itself and the words before it: the later scores are
set to −∞ before the softmax, so they get a share of exactly 0. The full 13 × 13 pattern of this head:

In [ ]:
pat = c["pattern"][L_, H_].cpu().numpy()
fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.imshow(pat ** 0.5, cmap="gray_r", vmin=0, vmax=1)
ax.set_xticks(range(T), WORDS, rotation=90, fontsize=8); ax.set_yticks(range(T), WORDS, fontsize=8)
ax.set_xlabel("looks at"); ax.set_ylabel("word"); ax.set_title("Block 11, head 12 (ink: share, square-rooted)", fontsize=10)
plt.show()
check("nothing above the diagonal", float(np.triu(pat, 1).sum()), 0.0)

### 9. Values, mixed by the pattern (Plate X)
The head hands the last word a **mix** of the 13 values, each weighted by its share: 64 numbers, mostly *for*'s value.

In [ ]:
z = share @ V
gap = (z - (c["pattern"][L_, H_, -1] @ c["v"][L_, H_])).abs().max().item()
print("the mix, first 4 numbers:", np.round(z[:4].tolist(), 4))
print(f"against the model's own mix: largest difference {gap:.1e}")
check("the mix, against the model's own (largest difference)", gap, 2.4e-07, tol=1e-5)

### 10. Writing back (Plate XI)
W_O turns the 64-number mix into 768 numbers, which are **added** to the last word's running total. Read through the
word table (Lecture 19's pushes, with the final LayerNorm's spread frozen), how much does this addition push walk above
the average word? And what would the head write if it looked at *for* alone?

In [ ]:
out = z @ W_O
print(f"the head adds an arrow of length {out.norm():.1f} to a running total of length {c['resid'][L_, -1].norm():.1f}")
check("the addition's length", out.norm().item(), 20.9378, tol=0.01)
check("the running total's length before", c["resid"][L_, -1].norm().item(), 160.3423, tol=0.02)

xf = c["resid"][12, -1]
sigma = torch.sqrt(((xf - xf.mean()) ** 2).mean() + 1e-5)
gain = SD["ln_f.weight"]
u_walk = WTE[WALK] - WTE.mean(0)
def push(v, u=u_walk):
    """What an addition v to the last word's total does to (a word's score − the average word's)."""
    return (((v - v.mean()) / sigma * gain) @ u).item()

own = c["head_out"][L_, H_, -1]                          # the model's own copy (W_O's bias is shared by all heads)
print(f"its push on walk, above the average word: {push(own):+.2f}")
check("block 11, head 12 pushes walk by", push(own), 0.9499, tol=2e-3)
each = (x @ W_V + b_V) @ W_O                             # what it would write from each word, looked at alone
read = ((each - each.mean(-1, keepdim=True)) / sigma * gain) @ WTE.T
read = read - read.mean(-1, keepdim=True)
top_for = [decode([j]) for j in read[11].topk(5).indices.tolist()]
print("from 'for' alone, it would push:", top_for)
check("the words it pushes from 'for'", top_for, ['trip', ' trip', ' trips', ' stroll', ' voyage'])

### 11. Where to look, and what to move (Plate XII)
A head does two separate jobs, with separate weights: **where to look** (W_Q and W_K: the *QK circuit*) and **what to
move** from the word it looks at (W_V and W_O: the *OV circuit*). The OV side, word by word: each word's push on walk,
if the head looked only there.

In [ ]:
ov_walk = [push(e) for e in each]
for w, v in zip(WORDS, ov_walk):
    print(f"  {w!r:9} {v:+.2f}")
check("each word's push through the OV side (largest difference)", max(abs(a - b) for a, b in zip(ov_walk, [-0.0214, 0.1137, 0.3165, 0.0855, 0.0183, 0.6553, 0.721, 0.2638, -0.0929, 0.2776, 0.6754, 1.0832, 0.8807])), 0, tol=3e-3)

## Part 3 · All 144 heads

### 12. 144 heads (Plate XIII)
Every head of every block, drawn small. Most do the same thing on this sentence: they park most of their attention on
the first word, *It*.

In [ ]:
P = c["pattern"].cpu().numpy()                                  # 12 blocks x 12 heads x 13 x 13
on_it = P[:, :, 1:, 0].mean(-1)                                 # each head's average share on "It" (words 2-13)
print(f"{(on_it > 0.5).sum()} of 144 heads put more than half their attention on 'It'")
check("heads parked on 'It'", int((on_it > 0.5).sum()), 104)
check("every head's share on 'It' (largest difference)", float(np.abs(on_it - np.array([[0.388, 0.003, 0.391, 0.084, 0.215, 0.082, 0.336, 0.214, 0.185, 0.355, 0.314, 0.363], [0.318, 0.373, 0.372, 0.525, 0.546, 0.432, 0.558, 0.604, 0.56, 0.657, 0.001, 0.17], [0.41, 0.766, 0.151, 0.28, 0.375, 0.249, 0.644, 0.286, 0.235, 0.261, 0.419, 0.73], [0.946, 0.618, 0.245, 0.514, 0.953, 0.642, 0.3, 0.278, 0.388, 0.402, 0.712, 0.302], [0.418, 0.538, 0.858, 0.465, 0.753, 0.661, 0.694, 0.334, 0.689, 0.492, 0.895, 0.083], [0.957, 0.998, 0.652, 0.548, 0.625, 0.98, 0.669, 0.796, 0.889, 0.95, 0.705, 0.741], [0.551, 0.743, 0.817, 0.741, 0.629, 0.733, 0.836, 0.605, 0.592, 0.995, 0.919, 0.604], [0.619, 0.962, 0.999, 0.76, 0.903, 0.679, 0.878, 0.976, 0.612, 0.705, 0.996, 0.975], [0.847, 0.99, 0.756, 0.869, 0.651, 0.597, 0.825, 0.518, 0.6, 0.831, 0.707, 0.828], [0.819, 0.964, 0.857, 0.524, 0.956, 0.871, 0.987, 0.853, 0.908, 0.991, 0.619, 0.979], [0.853, 0.974, 0.975, 0.9, 0.746, 0.843, 0.965, 0.754, 0.974, 0.793, 0.968, 0.787], [0.226, 0.892, 0.926, 0.578, 0.851, 0.95, 0.964, 0.899, 0.0, 0.972, 0.666, 0.759]])).max()), 0, tol=2e-3)
fig, ax = plt.subplots(12, 12, figsize=(8, 8))
for l in range(12):
    for h in range(12):
        ax[l, h].imshow(P[l, h] ** 0.5, cmap="gray_r", vmin=0, vmax=1); ax[l, h].set_xticks([]); ax[l, h].set_yticks([])
        if h == 0: ax[l, h].set_ylabel(l + 1, fontsize=7, rotation=0, labelpad=8)
        if l == 0: ax[l, h].set_title(h + 1, fontsize=7)
fig.suptitle("144 heads on our sentence (rows: blocks; columns: heads)", fontsize=10); plt.show()

### 13. A head that looks one word back (Plate XIV)
Block 5, head 12 does something else: every word looks at the word just before it. Its **look-back score** is the
average share on the previous word. Is it about meaning? Test it on 20 sequences of 50 random tokens, each repeated
twice (the lecture's own random tokens: NumPy's generator with seed 21).

In [ ]:
prev = np.array([[P[l, h].diagonal(-1).mean() for h in range(12)] for l in range(12)])
print(f"block 5, head 12 on our sentence: {prev[4, 11]:.3f}")
check("its look-back score on our sentence", prev[4, 11], 1.0, tol=2e-3)
rng = np.random.default_rng(21)
prev_r = np.zeros((12, 12))
for _ in range(20):
    r = rng.integers(1000, 30000, size=50)
    cr = run(np.concatenate([[50256], r, r]).tolist())
    prev_r += cr["pattern"].diagonal(-1, dim1=-2, dim2=-1).mean(-1).cpu().numpy() / 20
l, h = divmod(int(prev_r.argmax()), 12)
print(f"on random tokens, the strongest look-back head is block {l + 1}, head {h + 1}: {prev_r[l, h]:.3f}")
check("on random tokens, block 5 head 12", prev_r[4, 11], 0.987, tol=2e-3)
check("...the strongest look-back head of all 144", [l, h], [4, 11])

### 14. Heads that push walk (Plate XV)
Each head's addition at the last word, read as a push on walk (Step 10's `push`), for all 144.

In [ ]:
D = np.array([[push(c["head_out"][l, h, -1]) for h in range(12)] for l in range(12)])
order = np.argsort(-D.ravel())
for i in list(order[:3]) + [int(D.argmin())]:
    l, h = divmod(int(i), 12)
    print(f"  block {l + 1:2}, head {h + 1:2}: {D[l, h]:+.2f}")
check("the three heads pushing walk hardest (from 0)", [list(divmod(int(i), 12)) for i in order[:3]], [[11, 3], [10, 11], [9, 3]])
check("their pushes (largest difference)", max(abs(D[l, h] - v) for (l, h), v in zip([[11, 3], [10, 11], [9, 3]], [1.0517, 0.9499, 0.3965])), 0, tol=3e-3)
check("the head pushing hardest against walk (from 0)", list(divmod(int(D.argmin()), 12)), [8, 10])
check("all 144 pushes (largest difference)", float(np.abs(D - np.array([[0.0342, 0.0938, 0.0084, 0.0218, -0.032, 0.1059, 0.0353, 0.0654, 0.0513, 0.1381, -0.059, 0.0114], [-0.0629, 0.1366, -0.0329, 0.1603, 0.0549, -0.016, 0.0226, 0.0328, -0.0183, -0.0094, -0.1217, -0.0555], [-0.136, 0.0077, -0.0599, -0.1205, -0.0356, 0.0693, 0.0393, 0.0429, 0.0609, 0.0948, 0.0023, 0.0817], [-0.0705, -0.0078, -0.0461, 0.0129, 0.0795, 0.0414, -0.1049, 0.0897, 0.049, 0.0278, 0.0007, 0.0119], [0.0322, -0.0246, -0.0212, -0.0086, -0.0591, 0.0006, 0.0178, 0.0152, 0.0766, 0.0389, 0.0353, 0.0025], [-0.036, 0.068, 0.0056, 0.0516, 0.0046, -0.0499, 0.0803, 0.0601, -0.0922, 0.1213, 0.0855, -0.0489], [0.0397, -0.0091, 0.0045, -0.0196, 0.1493, -0.0719, -0.0787, 0.1221, -0.0138, 0.0039, 0.0229, 0.0881], [0.0964, 0.003, -0.0044, -0.0479, 0.0214, 0.1116, -0.01, -0.0065, 0.2037, 0.0334, 0.0162, 0.0107], [0.0034, 0.019, 0.0766, 0.0288, 0.2128, 0.1937, -0.0094, 0.1378, 0.2768, 0.016, -0.4243, 0.0854], [0.0684, -0.0007, 0.0175, 0.3965, -0.0008, -0.1845, 0.0362, -0.0357, -0.0251, 0.0098, 0.0838, -0.005], [0.0727, 0.0276, 0.052, -0.0132, 0.1348, 0.0045, 0.0103, -0.1688, -0.006, 0.0904, 0.0341, 0.9499], [0.1079, 0.0864, 0.0928, 1.0517, -0.003, -0.162, 0.0134, 0.0541, 0.1593, 0.0166, 0.1446, 0.2314]])).max()), 0, tol=3e-3)
fig, ax = plt.subplots(figsize=(4.4, 4))
im = ax.imshow(D, cmap="gray_r"); ax.set_xlabel("head"); ax.set_ylabel("block")
ax.set_xticks(range(12), range(1, 13), fontsize=7); ax.set_yticks(range(12), range(1, 13), fontsize=7)
plt.colorbar(im, shrink=0.8).set_label("push on walk"); plt.show()

### 15. The attention sink (Plate XVI)
If *It* means so little, why do 104 heads look at it? Because a head's shares must add up to 1, and a head with nothing
useful to fetch needs somewhere harmless to put them. *It*'s value is nearly empty, so parking there adds almost
nothing. Hide *It*, and those heads must spread their attention over words that do carry something.

In [ ]:
vn = V.norm(dim=-1)
print(f"block 11, head 12: the value of 'It' has length {vn[0]:.2f}; the value of 'for' {vn[11]:.2f}")
check("the length of 'It''s value", vn[0].item(), 0.7414, tol=2e-3)
check("the length of 'for''s value", vn[11].item(), 7.616, tol=5e-3)
print(f"hide 'It' (Step 2): walk falls from {100 * P_WALK:.0f}% to {100 * hide[1]:.1f}%")
check("walk with 'It' hidden", hide[1], 0.00205, tol=5e-5)

### 16. Read from, and written to, one stream (Plates XVII and XIX)
Every head and every neuron reads the running total and adds to it. So the whole of GPT-2 small is: two tables, twelve
blocks of (LayerNorm, 12 heads, LayerNorm, 3,072 neurons), and a last LayerNorm. Count its weights part by part.

In [ ]:
n = lambda *keys: sum(SD[k].numel() for k in keys)
part = lambda p: sum(v.numel() for k, v in SD.items() if k.startswith(f"h.0.{p}"))
census = {"word table": n("wte.weight"), "position table": n("wpe.weight"),
          "12 × attention": 12 * part("attn."), "12 × neurons": 12 * part("mlp."),
          "25 LayerNorms": 12 * (part("ln_1.") + part("ln_2.")) + n("ln_f.weight", "ln_f.bias")}
for k, v in census.items():
    print(f"  {k:16} {v:>12,}")
print(f"  {'in all':16} {sum(census.values()):>12,}")
check("one block's attention", part("attn."), 2362368)
check("one block's neurons", part("mlp."), 4722432)
check("every weight of GPT-2 small", sum(census.values()), 124439808)

### 17. Switching heads off (Plate XVIII)
Zero one head's output (it then adds nothing) and see what happens to walk; all 144, one at a time. Two heads stand
out. Note that "zero" is not a neutral *off*: a head that usually adds something is now adding nothing at all.

In [ ]:
Z = np.array([[p_walk(zero_heads=[(l, h)]) for h in range(12)] for l in range(12)])
hi, lo = divmod(int(Z.argmax()), 12), divmod(int(Z.argmin()), 12)
print(f"most helpful when off: block {hi[0] + 1}, head {hi[1] + 1} → walk {100 * Z[hi]:.1f}%")
print(f"most harmful when off: block {lo[0] + 1}, head {lo[1] + 1} → walk {100 * Z[lo]:.2f}%")
check("all 144 heads zeroed one by one (largest difference)", float(np.abs(Z - np.array([[0.54538, 0.57289, 0.53002, 0.58004, 0.58303, 0.57778, 0.58055, 0.46984, 0.5175, 0.54824, 0.00222, 0.57983], [0.54639, 0.55742, 0.57921, 0.5799, 0.56782, 0.58218, 0.58806, 0.5237, 0.57017, 0.57838, 0.57033, 0.57782], [0.64207, 0.5814, 0.52429, 0.58967, 0.60498, 0.56976, 0.58247, 0.58326, 0.61164, 0.56402, 0.5307, 0.59376], [0.49112, 0.61324, 0.5676, 0.5695, 0.63393, 0.54762, 0.6, 0.56057, 0.61986, 0.5505, 0.58944, 0.52439], [0.58732, 0.58729, 0.59446, 0.56175, 0.61945, 0.57824, 0.55703, 0.5751, 0.52325, 0.57769, 0.57523, 0.53832], [0.55816, 0.75542, 0.58271, 0.59355, 0.57089, 0.49785, 0.55526, 0.56713, 0.54946, 0.59307, 0.56896, 0.55789], [0.58491, 0.57955, 0.58068, 0.56302, 0.61451, 0.56967, 0.5741, 0.5458, 0.55152, 0.58235, 0.56859, 0.59008], [0.58715, 0.56662, 0.59029, 0.59024, 0.58821, 0.54122, 0.57292, 0.57714, 0.55653, 0.54842, 0.57564, 0.57461], [0.5699, 0.57858, 0.56999, 0.57732, 0.58638, 0.53212, 0.57621, 0.61802, 0.55292, 0.58516, 0.58225, 0.56527], [0.58479, 0.59355, 0.57454, 0.59757, 0.57016, 0.57841, 0.57166, 0.58999, 0.58459, 0.57996, 0.58175, 0.57877], [0.54903, 0.60169, 0.5799, 0.59934, 0.57412, 0.57929, 0.57629, 0.56816, 0.57917, 0.57249, 0.56679, 0.55111], [0.47524, 0.56914, 0.56783, 0.54602, 0.56344, 0.60603, 0.58662, 0.57297, 0.58293, 0.57983, 0.56336, 0.55926]])).max()), 0, tol=3e-4)
check("the head whose loss raises walk most (from 0)", list(hi), [5, 1])
check("the head whose loss sinks walk (from 0)", list(lo), [0, 10])
pz = torch.softmax(run(ids, zero_heads=[(5, 1)])["logits"][-1], -1)
print("block 6, head 2 off, the top words:", [decode([j]) for j in pz.topk(3).indices.tolist()])
layers = [p_walk(zero_heads=[(l, h) for h in range(12)]) for l in range(12)]
print("each block's whole attention zeroed:", [f"{100 * v:.0f}%" for v in layers])
check("each block's attention zeroed (largest difference)", max(abs(a - b) for a, b in zip(layers, [0.47833, 0.54162, 0.55926, 0.55894, 0.5277, 0.52427, 0.54552, 0.48518, 0.53458, 0.61689, 0.5293, 0.37133])), 0, tol=3e-4)

## Part 4 · Smaller models

### 18. No blocks at all (Plate XX)
What can a transformer do with **no** blocks? Each word's arrow goes straight to LayerNorm and the word table: the next
word can depend only on the current one. The best it can learn is a table of which word follows which, the
**bigram** table. The lecture trained such a model on 983 million tokens of FineWeb (web text) and compared its guesses
with the real counts in 100 million of its training tokens. Count them yourself: the course used a public copy of FineWeb
already cut into GPT-2's tokens, 100 million per file (200 MB, 2 bytes per token).

In [ ]:
fetch("https://huggingface.co/datasets/kjj0/fineweb10B-gpt2/resolve/main/fineweb_train_000001.bin", "fineweb_1.bin",
      "100 million FineWeb tokens")
raw = open("fineweb_1.bin", "rb").read()
head_ = np.frombuffer(raw[:1024], dtype=np.int32)                # a 1,024-byte header: a magic number, a version, the count
SHARD = np.frombuffer(raw[1024:], dtype=np.uint16)[: head_[2]]
print(f"{len(SHARD):,} tokens")

def follows(word, k=5):
    """What follows a word in the 100 million tokens: the top k next tokens and their shares, and how often it appears."""
    t = encode(word)[0]
    at = np.nonzero(SHARD[:-1] == t)[0]
    nxt = np.bincount(SHARD[at + 1].astype(np.int64), minlength=50257)
    top = np.argsort(-nxt)[:k]
    return [(decode([int(i)]), round(float(nxt[i] / nxt.sum()), 4)) for i in top], len(at)

counts = {}
for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']:
    counts[w] = follows(w)
    print(f"after {w!r:12} ({counts[w][1]:>9,} times): " + ", ".join(f"{a!r} {100 * b:.1f}%" for a, b in counts[w][0][:3]))
check("the commonest next word after each of the 10 (word, share)", [tuple(counts[w][0][0]) for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']],
      [tuple({' United': {'top': [' States', 0.7081], 'n': 17531}, ' Barack': {'top': [' Obama', 0.9419], 'n': 1050}, ' New': {'top': [' York', 0.3673], 'n': 35089}, ' the': {'top': [' same', 0.0103], 'n': 3413372}, ' Mr': {'top': ['.', 0.7576], 'n': 10618}, ' according': {'top': [' to', 0.9874], 'n': 11669}, ' Los': {'top': [' Angeles', 0.825], 'n': 3937}, ' per': {'top': [' cent', 0.1416], 'n': 24162}, ' walk': {'top': [' in', 0.0675], 'n': 9221}, ' for': {'top': [' the', 0.153], 'n': 743923}}[w]["top"]) for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']])
check("how often each of the 10 appears", [counts[w][1] for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']], [{' United': {'top': [' States', 0.7081], 'n': 17531}, ' Barack': {'top': [' Obama', 0.9419], 'n': 1050}, ' New': {'top': [' York', 0.3673], 'n': 35089}, ' the': {'top': [' same', 0.0103], 'n': 3413372}, ' Mr': {'top': ['.', 0.7576], 'n': 10618}, ' according': {'top': [' to', 0.9874], 'n': 11669}, ' Los': {'top': [' Angeles', 0.825], 'n': 3937}, ' per': {'top': [' cent', 0.1416], 'n': 24162}, ' walk': {'top': [' in', 0.0675], 'n': 9221}, ' for': {'top': [' the', 0.153], 'n': 743923}}[w]["n"] for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']])

Now the model itself. The lecture's trained models are published with this notebook as *safetensors* files (Lecture 19's
format), in **bf16**, a 2-byte number format with float32's range and fewer digits, exactly as training saved them.
Each comes in two parts: everything but the last table, and the last table (50,257 × 256). We check each file's
fingerprint (its SHA-256) before using it.

In [ ]:
import hashlib
SITE = "https://glass-box-sigma.vercel.app/lectures/20-attention/notebook/"
SHA = {'l20_0block_a.safetensors': '7f29cc11e1c6cdd2d3f106f59ef85d5d2405e723d4c83da1c0cf4c1380276f37', 'l20_0block_b.safetensors': '3fc01d07f922b25f80e269bb44e265d584721ff10bbafeee046f041f2e07bef5', 'l20_1block_a.safetensors': 'dca5c5a8d50154fffda0ffd2f56e20ddd1a374945888766770c5231c9f78ed70', 'l20_1block_b.safetensors': 'b216b8c909fd2f9bf30ae9147c9304839ce7f5f70dcdd5a6bfe7fd6f20b98e24'}

def read_weights(path):
    """A safetensors file, any number format, as float32 tensors on the GPU."""
    raw = bytearray(open(path, "rb").read())
    n = int.from_bytes(raw[:8], "little")
    head = json.loads(raw[8:8 + n])
    kinds = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32, "I32": torch.int32}
    out = {}
    for name, h in head.items():
        if name != "__metadata__":
            a, b = h["data_offsets"]
            t = torch.frombuffer(raw, dtype=kinds[h["dtype"]], count=(b - a) // kinds[h["dtype"]].itemsize, offset=8 + n + a)
            out[name] = t.reshape(h["shape"]).to(DEV).float()
    return out

def load_small(stem, what):
    sd = {}
    for part in "ab":
        fn = f"{stem}_{part}.safetensors"
        fetch(SITE + fn, fn, f"{what}, part {part}")
        assert hashlib.sha256(open(fn, "rb").read()).hexdigest() == SHA[fn], f"{fn} is not the published file"
        sd.update(read_weights(fn))
    print(f"{what} loaded, {sum(v.numel() for v in sd.values()):,} weights")
    return sd

def small_run(sd, ids):
    """Our attention-only transformer: word + position arrows, then blocks of (LayerNorm -> 8 heads of 32 -> add),
    then LayerNorm and the last table. No neurons and no biases in attention. Returns scores and attention patterns."""
    ids = torch.as_tensor(ids, device=DEV); T_ = len(ids)
    x = sd["E.weight"][ids] + sd["P.weight"][:T_]
    mask = torch.tril(torch.ones(T_, T_, dtype=torch.bool, device=DEV))
    pats = []
    for l in range(sum(k.startswith("qkv.") for k in sd)):
        h = layer_norm(x, sd[f"ln.{l}.weight"], sd[f"ln.{l}.bias"])
        q, k, v = (h @ sd[f"qkv.{l}.weight"].T).split(256, -1)
        q, k, v = (t.view(T_, 8, 32).transpose(0, 1) for t in (q, k, v))
        a = torch.softmax((q @ k.transpose(-1, -2) / math.sqrt(32)).masked_fill(~mask, float("-inf")), -1)
        pats.append(a)
        x = x + (a @ v).transpose(0, 1).reshape(T_, 256) @ sd[f"o.{l}.weight"].T
    return layer_norm(x, sd["lnf.weight"], sd["lnf.bias"]) @ sd["U.weight"].T, pats

M0 = load_small("l20_0block", "0-block model")
guess = {}
for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']:
    p = torch.softmax(small_run(M0, [50256] * 99 + encode(w))[0][-1], -1)
    guess[w] = decode([int(p.argmax())])
    print(f"after {w!r:12} the 0-block model guesses {guess[w]!r} ({100 * p.max():.1f}%); the counts say {counts[w][0][0][0]!r} ({100 * counts[w][0][0][1]:.1f}%)")
check("the 0-block model's guess after each of the 10", [guess[w] for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']], [{' United': ' States', ' Barack': ' Obama', ' New': ' York', ' the': ' same', ' Mr': '.', ' according': ' to', ' Los': ' Angeles', ' per': ' cent', ' walk': ' in', ' for': ' the'}[w] for w in [' United', ' Barack', ' New', ' the', ' Mr', ' according', ' Los', ' per', ' walk', ' for']])

With no blocks, it cannot look back at all. Test it as Lecture 21 does: 256 random tokens, then the same 256 again
(64 such sequences, from the lecture's own random numbers). A model that could look back would find the second copy
easy; this one finds it exactly as hard as the first.

In [ ]:
rng = np.random.default_rng(1234)
rng.integers(0, 20_000_000 - 1026, size=256)                     # (the lecture drew its text samples first)
toks = rng.integers(1000, 30000, size=(64, 256))
REP = [[50256] + list(r) + list(r) for r in toks]

def rep_losses(sd):
    first, second = [], []
    for seq in REP:
        lp = torch.log_softmax(small_run(sd, seq)[0][:-1], -1).gather(-1, torch.tensor(seq[1:], device=DEV)[:, None])[:, 0]
        first.append(-lp[1:256].mean().item()); second.append(-lp[256:].mean().item())
    return np.mean(first), np.mean(second)

r0 = rep_losses(M0)
print(f"0 blocks, random tokens: {r0[0]:.2f} nats the first time, {r0[1]:.2f} the second")
check("0 blocks: loss on the first copy", r0[0], [14.149527549743652, 14.150733947753906][0], tol=0.02)
check("0 blocks: loss on the second copy", r0[1], [14.149527549743652, 14.150733947753906][1], tol=0.02)

### 19. One block of heads (Plate XXI)
Give the model one block of 8 heads. Elhage et al. (2021) found that most heads of such models **copy**: whatever word a
head looks at, it raises that same word's chance. Read it straight from the weights: a head's OV circuit, followed
through to the scores, is a 50,257 × 50,257 table (from the word looked at, to the word pushed). It copies if that table
mostly pushes each word onto itself. A single number for it: the table's **eigenvalues** (the amounts by which it
stretches its special directions) are mostly positive for a copier. The score is the sum of their real parts over the
sum of their sizes: +1 for a pure copier, −1 for a pure anti-copier. (The big table has the same nonzero eigenvalues as
a small 32 × 32 one, which is what we compute.)

In [ ]:
M1 = load_small("l20_1block", "1-block model")
E1 = layer_norm(M1["E.weight"] + M1["P.weight"][:512].mean(0), M1["ln.0.weight"], M1["ln.0.bias"])   # every token, as the block sees it
UG = M1["U.weight"] * M1["lnf.weight"]
copy = []
for h in range(8):
    W_V1 = M1["qkv.0.weight"][512 + 32 * h: 512 + 32 * (h + 1)].T            # 256 x 32
    W_O1 = M1["o.0.weight"][:, 32 * h: 32 * (h + 1)].T                       # 32 x 256
    ev = torch.linalg.eigvals(W_O1 @ (UG.T @ E1) @ W_V1)
    copy.append((ev.real.sum() / ev.abs().sum()).item())
print("copying score of each head:", [f"{v:+.3f}" for v in copy])
check("each head's copying score (largest difference)", max(abs(a - b) for a, b in zip(copy, [-0.0459, 0.9994, 0.9415, 0.7892, 0.0669, 0.9973, 0.8622, 0.9827])), 0, tol=5e-3)
check("heads that copy (score above 0.75)", sum(v > 0.75 for v in copy), sum(v > 0.75 for v in [-0.0459, 0.9994, 0.9415, 0.7892, 0.0669, 0.9973, 0.8622, 0.9827]))
r1 = rep_losses(M1)
print(f"1 block, random tokens: {r1[0]:.2f} nats the first time, {r1[1]:.2f} the second: barely easier")
check("1 block: loss on the first copy", r1[0], [13.742770195007324, 13.68875503540039][0], tol=0.02)
check("1 block: loss on the second copy", r1[1], [13.742770195007324, 13.68875503540039][1], tol=0.02)

### 20. What one block cannot do (Plate XXII)
Lecture 21's passage names a man twice: *Mr Quillon Barraclough*. GPT-2 copies the name the second time. One block of
copying heads cannot: to copy *illon* after *Qu*, a head must find "the word that came after the earlier *Qu*", and that
needs a head in an earlier block to have marked each word with the word before it. With one block there is no earlier
block.

In [ ]:
PASSAGE = ("Every evening the lighthouse keeper, Mr Quillon Barraclough, climbed the hundred and twelve steps to light the "
           "lamp. In the village they liked to say that Mr Quillon Barraclough had not missed a single night in forty years.")
pas = [50256] + encode(PASSAGE)
second = [i for i in range(len(pas)) if decode(pas[i:i + 2]) == " Quillon"][1]
def name_chances(logits):
    p = torch.softmax(logits, -1)
    return [p[i - 1, pas[i]].item() for i in range(second, second + 6)]
g2 = name_chances(run(pas)["logits"])
o1 = name_chances(small_run(M1, pas)[0])
for i, (a, b) in enumerate(zip(g2, o1)):
    print(f"  {decode([pas[second + i]])!r:8} GPT-2 {100 * a:5.1f}%    1 block {100 * b:5.2f}%")
check("GPT-2, the name's six pieces the second time (largest difference)", max(abs(a - b) for a, b in zip(g2, [0.3897, 0.9962, 0.0213, 0.9962, 0.9986, 0.9988])), 0, tol=2e-3)
check("GPT-2 gives more than 99% to", sum(v > 0.99 for v in g2), 4)
check("the 1-block model, its best for any piece after 'Qu'", max(o1[1:]), 0.0325, tol=5e-3)

## Part 5 · Pictures as tokens

### 21. A picture as tokens (Plate XXIII)
A **vision transformer** reads a photo the way GPT-2 reads text: cut it into square patches, turn each patch into an
arrow, add a position arrow, and add one extra **summary token** in front, whose final total is used to describe the
whole picture. Its attention has no mask: every token may look at every other.

The lecture used **DINOv2 ViT-S/14** (Oquab et al., 2023; 14 × 14-pixel patches) on ImageNet validation photos, cut to
224 × 224, so 16 × 16 = 256 patches. Here the model comes from Hugging Face through the `transformers` library, as in the
lecture; the photos are the lecture's own crops, as published on the course site.

In [ ]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"               # the page shows results, not download bars
import transformers
from PIL import Image
transformers.logging.set_verbosity_error()
PHOTOS = [['goldfinch', 'img/vit_00.jpg'], ['goldfinch', 'img/vit_01.jpg'], ['hummingbird', 'img/vit_03.jpg'], ['hummingbird', 'img/vit_04.jpg'], ['golden retriever', 'img/vit_06.jpg'], ['golden retriever', 'img/vit_07.jpg'], ['tabby', 'img/vit_09.jpg'], ['daisy', 'img/vit_25.jpg']]                                            # (name, its file on the course site)
ims = []
for name, img in PHOTOS:
    fetch("https://glass-box-sigma.vercel.app/lectures/20-attention/" + img, img.split("/")[-1], f"photo {img.split('/')[-1]} ({name})")
    ims.append(Image.open(img.split("/")[-1]).convert("RGB"))
PHOTOS = [name for name, _ in PHOTOS]
mean, std = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1), torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
X = torch.stack([(torch.tensor(np.asarray(im), dtype=torch.float32).permute(2, 0, 1) / 255 - mean) / std for im in ims]).to(DEV)
vit = {name: transformers.AutoModel.from_pretrained(name, attn_implementation="eager").to(DEV).eval()
       for name in ("facebook/dinov2-small", "facebook/dinov2-with-registers-small")}
cfg = vit["facebook/dinov2-small"].config
n_w = sum(p.numel() for p in vit["facebook/dinov2-small"].parameters())
print(f"DINOv2 ViT-S/14: {cfg.num_hidden_layers} blocks, {cfg.num_attention_heads} heads, {cfg.hidden_size} numbers per token, {n_w:,} weights")
check("its blocks", cfg.num_hidden_layers, 12)
check("its heads", cfg.num_attention_heads, 6)
check("its weights", n_w, 22056576)
fig, ax = plt.subplots(1, 8, figsize=(11, 1.7))
for a, im, name in zip(ax, ims, PHOTOS):
    a.imshow(im); a.set_title(name, fontsize=8); a.set_xticks([]); a.set_yticks([])
plt.show()

### 22. Where the summary token looks (Plate XXIV)
The last block's attention from the summary token to the 256 patches, head by head. First, to be sure the library's
attention is the one we know, recompute the last block's summary-token attention by hand from its weights.

In [ ]:
with torch.no_grad():
    o = vit["facebook/dinov2-small"](pixel_values=X, output_attentions=True, output_hidden_states=True)
att = torch.stack(o.attentions)                                  # blocks x photos x heads x 257 x 257
blk = vit["facebook/dinov2-small"].encoder.layer[-1]
with torch.no_grad():
    h_ = blk.norm1(o.hidden_states[-2])                          # the last block's input, LayerNormed
    sa = blk.attention.attention
    q_ = sa.query(h_[:, :1]).view(len(ims), 1, 6, 64).transpose(1, 2)        # the summary token's query, 6 heads
    k_ = sa.key(h_).view(len(ims), -1, 6, 64).transpose(1, 2)
    mine = torch.softmax(q_ @ k_.transpose(-1, -2) / 8, -1)[:, :, 0]         # photos x heads x 257
gap = (mine - att[-1, :, :, 0]).abs().max().item()
print(f"the summary token's attention, by hand against the library's: largest difference {gap:.1e}")
assert gap < 1e-4

cls = att[-1, :, :, 0, 1:].cpu().numpy()                         # photos x 6 heads x 256 patches
def half(r):
    s = np.sort(r / r.sum())[::-1]
    return int(np.searchsorted(np.cumsum(s), 0.5) + 1)
counts = [half(r) for r in cls[0]]
print("the goldfinch: patches holding half of each head's attention:", counts)
check("the goldfinch, head by head (largest difference in patches)", max(abs(a - b) for a, b in zip(counts, [4, 3, 2, 58, 4, 6])), 0, tol=3)
check("the goldfinch's map, all 6 heads (largest difference)", float(np.abs(cls[0] - np.array([[0.0005, 0.0001, 0.0001, 0.0005, 0.0001, 0.0004, 0.0004, 0.0009, 0.0003, 0.0004, 0.0008, 0.0002, 0.0001, 0.0004, 0.0001, 0.0003, 0.0008, 0.0002, 0.0006, 0.0009, 0.0002, 0.0, 0.0015, 0.0013, 0.0, 0.0002, 0.0005, 0.0001, 0.0002, 0.0012, 0.0003, 0.0, 0.0005, 0.0001, 0.0001, 0.0001, 0.0001, 0.0001, 0.0001, 0.0002, 0.0, 0.001, 0.0023, 0.0011, 0.0002, 0.0003, 0.0001, 0.0001, 0.0013, 0.0016, 0.0002, 0.0009, 0.0002, 0.0009, 0.0003, 0.0001, 0.0002, 0.0009, 0.0003, 0.0006, 0.0011, 0.0007, 0.0002, 0.0008, 0.0006, 0.0001, 0.0003, 0.0001, 0.0001, 0.0003, 0.0019, 0.0029, 0.0001, 0.0001, 0.0003, 0.0002, 0.0007, 0.0008, 0.0006, 0.0005, 0.0003, 0.0004, 0.0001, 0.0005, 0.0001, 0.0003, 0.0041, 0.0095, 0.0168, 0.0001, 0.0, 0.0001, 0.0016, 0.0005, 0.0002, 0.0006, 0.0005, 0.0003, 0.0006, 0.0006, 0.0, 0.0006, 0.0006, 0.0473, 0.068, 0.0005, 0.001, 0.0003, 0.0001, 0.0001, 0.0001, 0.0001, 0.0005, 0.0014, 0.0003, 0.0009, 0.0001, 0.0005, 0.0275, 0.2557, 0.116, 0.0238, 0.0002, 0.0002, 0.0001, 0.0006, 0.0013, 0.0004, 0.0006, 0.0002, 0.0004, 0.0004, 0.0002, 0.0007, 0.0002, 0.0874, 0.0521, 0.0002, 0.0007, 0.0001, 0.0006, 0.0002, 0.001, 0.0003, 0.0006, 0.0011, 0.0003, 0.0002, 0.0003, 0.0015, 0.001, 0.0274, 0.0509, 0.0003, 0.0007, 0.0013, 0.0006, 0.0004, 0.0002, 0.0005, 0.0008, 0.0004, 0.0009, 0.0002, 0.0005, 0.0003, 0.0001, 0.0012, 0.0093, 0.0067, 0.001, 0.0019, 0.0019, 0.0007, 0.0016, 0.0003, 0.0011, 0.0003, 0.0002, 0.0006, 0.0001, 0.0001, 0.0004, 0.0011, 0.0003, 0.0002, 0.0002, 0.0001, 0.0037, 0.0015, 0.0012, 0.0008, 0.0009, 0.0005, 0.0004, 0.0006, 0.0004, 0.0002, 0.0004, 0.0017, 0.0015, 0.0002, 0.0009, 0.0008, 0.002, 0.0005, 0.0002, 0.0011, 0.0006, 0.0002, 0.0004, 0.0003, 0.0003, 0.0016, 0.0007, 0.0003, 0.0014, 0.0003, 0.0001, 0.0018, 0.0013, 0.0015, 0.0004, 0.0003, 0.0009, 0.0007, 0.0003, 0.0004, 0.0007, 0.001, 0.0012, 0.0006, 0.0016, 0.002, 0.0002, 0.0005, 0.0007, 0.003, 0.0003, 0.0003, 0.001, 0.0005, 0.0006, 0.0005, 0.0006, 0.0003, 0.0002, 0.0002, 0.0002, 0.0002, 0.0, 0.0004, 0.0005, 0.0006, 0.0007, 0.0007], [0.0004, 0.0002, 0.0004, 0.0011, 0.0002, 0.0005, 0.0002, 0.0005, 0.0002, 0.0002, 0.0002, 0.0001, 0.0001, 0.0002, 0.0003, 0.0007, 0.0002, 0.0005, 0.0009, 0.0019, 0.0003, 0.0001, 0.0007, 0.0008, 0.0001, 0.0001, 0.0003, 0.0001, 0.0001, 0.0004, 0.0002, 0.0001, 0.0007, 0.0002, 0.0004, 0.0002, 0.0002, 0.0002, 0.0002, 0.0004, 0.0001, 0.0006, 0.0016, 0.0008, 0.0001, 0.0003, 0.0001, 0.0001, 0.0006, 0.0016, 0.0005, 0.002, 0.0005, 0.0013, 0.0005, 0.0003, 0.0001, 0.0005, 0.0004, 0.0003, 0.0003, 0.0002, 0.0001, 0.0003, 0.0002, 0.0002, 0.001, 0.0004, 0.0002, 0.0002, 0.0017, 0.0083, 0.0001, 0.0, 0.0001, 0.0001, 0.0003, 0.0007, 0.0003, 0.0003, 0.0002, 0.001, 0.0, 0.0002, 0.0001, 0.0003, 0.0259, 0.3439, 0.0395, 0.0, 0.0, 0.0, 0.0004, 0.0004, 0.0001, 0.0004, 0.0005, 0.0005, 0.0001, 0.0001, 0.0, 0.0001, 0.0067, 0.0287, 0.0383, 0.0005, 0.0003, 0.0002, 0.0002, 0.0, 0.0001, 0.0003, 0.0003, 0.0004, 0.0003, 0.0002, 0.0, 0.0001, 0.006, 0.0218, 0.0303, 0.0035, 0.0001, 0.0, 0.0001, 0.0014, 0.0033, 0.0009, 0.0001, 0.0003, 0.0001, 0.001, 0.0001, 0.0002, 0.0001, 0.0144, 0.0065, 0.0002, 0.0018, 0.0002, 0.0001, 0.0, 0.0002, 0.0001, 0.0004, 0.0002, 0.0003, 0.0001, 0.0001, 0.0037, 0.0028, 0.0939, 0.0327, 0.0001, 0.0001, 0.0003, 0.0002, 0.0003, 0.0001, 0.0001, 0.0006, 0.0002, 0.0028, 0.0005, 0.0002, 0.0001, 0.0, 0.0002, 0.0575, 0.036, 0.0003, 0.0006, 0.0005, 0.0003, 0.0004, 0.0001, 0.0032, 0.0001, 0.0003, 0.0004, 0.0002, 0.0, 0.0001, 0.0002, 0.0003, 0.0001, 0.0002, 0.0001, 0.0014, 0.0009, 0.0005, 0.0004, 0.0005, 0.0001, 0.0002, 0.0002, 0.0002, 0.0003, 0.0002, 0.0005, 0.0005, 0.0, 0.0008, 0.0005, 0.0011, 0.0002, 0.0002, 0.0008, 0.0002, 0.0003, 0.0001, 0.0002, 0.0001, 0.0003, 0.0003, 0.0001, 0.0012, 0.0001, 0.0, 0.0005, 0.0006, 0.0005, 0.0002, 0.0002, 0.0003, 0.0005, 0.0001, 0.0007, 0.0003, 0.0004, 0.0005, 0.0002, 0.0003, 0.0009, 0.0001, 0.0003, 0.0003, 0.0007, 0.0003, 0.0003, 0.001, 0.0014, 0.0009, 0.0006, 0.0012, 0.0003, 0.0001, 0.0004, 0.0003, 0.0002, 0.0001, 0.0003, 0.0002, 0.0004, 0.0026, 0.0006], [0.0002, 0.0001, 0.0002, 0.0003, 0.0001, 0.0008, 0.0001, 0.0003, 0.0001, 0.0003, 0.0001, 0.0001, 0.0002, 0.0001, 0.0001, 0.0003, 0.0001, 0.0001, 0.0007, 0.0006, 0.0003, 0.0, 0.0002, 0.0005, 0.0, 0.0, 0.0007, 0.0002, 0.0001, 0.0002, 0.0001, 0.0001, 0.0007, 0.0001, 0.0001, 0.0001, 0.0001, 0.0001, 0.0002, 0.0004, 0.0, 0.0002, 0.0006, 0.0002, 0.0, 0.0001, 0.0004, 0.0001, 0.0003, 0.0005, 0.0002, 0.0005, 0.0003, 0.0004, 0.0001, 0.0001, 0.0001, 0.0003, 0.0001, 0.0001, 0.0001, 0.0002, 0.0002, 0.0001, 0.0001, 0.0001, 0.0007, 0.0002, 0.0001, 0.0002, 0.0007, 0.0046, 0.0002, 0.0001, 0.0001, 0.0001, 0.0002, 0.0009, 0.0002, 0.0001, 0.0002, 0.0015, 0.0, 0.0001, 0.0001, 0.0003, 0.2914, 0.2319, 0.0087, 0.0, 0.0, 0.0001, 0.0003, 0.0002, 0.0001, 0.0002, 0.0004, 0.0004, 0.0001, 0.0001, 0.0, 0.0001, 0.0132, 0.0123, 0.0122, 0.0002, 0.0001, 0.0001, 0.0001, 0.0, 0.0, 0.0002, 0.0001, 0.0002, 0.0003, 0.0002, 0.0, 0.0003, 0.0145, 0.0497, 0.0277, 0.008, 0.0001, 0.0, 0.0001, 0.0007, 0.0012, 0.0005, 0.0001, 0.0003, 0.0001, 0.0004, 0.0002, 0.0003, 0.0003, 0.0302, 0.0125, 0.0003, 0.0014, 0.0002, 0.0001, 0.0001, 0.0001, 0.0001, 0.0015, 0.0003, 0.0005, 0.0, 0.0, 0.0016, 0.0016, 0.014, 0.0494, 0.0002, 0.0002, 0.0002, 0.0002, 0.0005, 0.0001, 0.0001, 0.0008, 0.0002, 0.0013, 0.0003, 0.0001, 0.0001, 0.0001, 0.0001, 0.011, 0.0071, 0.0002, 0.0002, 0.0005, 0.0007, 0.0002, 0.0001, 0.002, 0.0001, 0.0001, 0.0003, 0.0001, 0.0, 0.0001, 0.0001, 0.0007, 0.0001, 0.0002, 0.0001, 0.0006, 0.0006, 0.0002, 0.0003, 0.0004, 0.0001, 0.0001, 0.0003, 0.0003, 0.0002, 0.0008, 0.0002, 0.0003, 0.0, 0.0012, 0.0002, 0.0004, 0.0002, 0.0001, 0.0008, 0.0001, 0.0003, 0.0002, 0.0002, 0.0001, 0.0003, 0.0003, 0.0004, 0.0005, 0.0001, 0.0001, 0.0003, 0.0002, 0.0001, 0.0002, 0.0002, 0.0002, 0.0003, 0.0002, 0.0002, 0.0004, 0.0004, 0.0009, 0.0002, 0.0003, 0.0005, 0.0001, 0.0002, 0.0003, 0.0002, 0.0002, 0.0001, 0.0007, 0.0006, 0.0003, 0.0003, 0.0004, 0.0002, 0.0001, 0.0002, 0.0002, 0.0003, 0.0001, 0.0004, 0.0001, 0.0002, 0.0013, 0.0007], [0.0058, 0.0024, 0.0029, 0.0134, 0.001, 0.0061, 0.0046, 0.0065, 0.004, 0.0042, 0.003, 0.0016, 0.0008, 0.0026, 0.0045, 0.0069, 0.002, 0.0042, 0.0023, 0.0215, 0.0069, 0.0008, 0.004, 0.0049, 0.0009, 0.0008, 0.0031, 0.0017, 0.0018, 0.0022, 0.0019, 0.0005, 0.0018, 0.0005, 0.0019, 0.003, 0.0029, 0.001, 0.0019, 0.0031, 0.0003, 0.0036, 0.0056, 0.0046, 0.0012, 0.0012, 0.0013, 0.0009, 0.0039, 0.0054, 0.0024, 0.0227, 0.001, 0.0036, 0.0013, 0.0052, 0.0009, 0.0019, 0.0016, 0.0015, 0.0031, 0.0036, 0.0016, 0.0035, 0.0017, 0.001, 0.0013, 0.0038, 0.0013, 0.0015, 0.005, 0.0027, 0.0017, 0.0007, 0.0022, 0.0015, 0.0025, 0.0032, 0.0019, 0.0023, 0.0027, 0.0055, 0.0005, 0.0016, 0.0018, 0.0022, 0.0042, 0.0048, 0.0044, 0.0007, 0.0006, 0.0006, 0.0033, 0.0018, 0.0015, 0.0034, 0.0045, 0.0028, 0.0014, 0.0023, 0.0004, 0.0031, 0.0015, 0.0041, 0.0072, 0.0015, 0.0036, 0.0026, 0.0038, 0.0008, 0.0015, 0.0015, 0.0024, 0.003, 0.0022, 0.0032, 0.0009, 0.0032, 0.0025, 0.0074, 0.0045, 0.002, 0.0015, 0.0011, 0.0013, 0.0094, 0.0173, 0.009, 0.0024, 0.0015, 0.0009, 0.0054, 0.0019, 0.0042, 0.0016, 0.007, 0.0033, 0.0016, 0.0124, 0.0014, 0.0022, 0.0011, 0.0025, 0.0012, 0.0035, 0.0037, 0.0016, 0.001, 0.0011, 0.0196, 0.0153, 0.0061, 0.0046, 0.0012, 0.0023, 0.0031, 0.0016, 0.0017, 0.0012, 0.0024, 0.0032, 0.0014, 0.0152, 0.0019, 0.0027, 0.0018, 0.0007, 0.0036, 0.004, 0.0029, 0.0031, 0.0035, 0.004, 0.0023, 0.0039, 0.0012, 0.0094, 0.0009, 0.0027, 0.0038, 0.0013, 0.0011, 0.0018, 0.002, 0.0029, 0.0011, 0.0012, 0.0006, 0.0059, 0.0039, 0.0026, 0.0018, 0.0045, 0.0027, 0.002, 0.0024, 0.0029, 0.0031, 0.0034, 0.0032, 0.0031, 0.0015, 0.0086, 0.0021, 0.0035, 0.0027, 0.0019, 0.003, 0.0022, 0.0027, 0.0019, 0.0014, 0.0019, 0.0031, 0.0023, 0.0016, 0.0034, 0.0014, 0.0012, 0.0023, 0.0028, 0.0031, 0.0012, 0.0013, 0.0037, 0.0044, 0.002, 0.0053, 0.0033, 0.0029, 0.0039, 0.0024, 0.0052, 0.005, 0.0016, 0.0026, 0.0029, 0.005, 0.0019, 0.0036, 0.0046, 0.0066, 0.0072, 0.0062, 0.0073, 0.0027, 0.0009, 0.0032, 0.0025, 0.0024, 0.0015, 0.0029, 0.0029, 0.0023, 0.0098, 0.0045], [0.0004, 0.0002, 0.0003, 0.0014, 0.0002, 0.0007, 0.0002, 0.0005, 0.0001, 0.0003, 0.0002, 0.0001, 0.0001, 0.0004, 0.0005, 0.0011, 0.0001, 0.0005, 0.0019, 0.0022, 0.0005, 0.0001, 0.0008, 0.001, 0.0001, 0.0001, 0.0002, 0.0001, 0.0001, 0.0007, 0.0003, 0.0001, 0.0008, 0.0002, 0.0002, 0.0003, 0.0003, 0.0002, 0.0003, 0.0004, 0.0001, 0.0007, 0.0012, 0.0009, 0.0001, 0.0002, 0.0001, 0.0001, 0.0004, 0.0008, 0.0003, 0.0026, 0.001, 0.0012, 0.0003, 0.0003, 0.0002, 0.0006, 0.0005, 0.0005, 0.0003, 0.0002, 0.0001, 0.0003, 0.0002, 0.0002, 0.001, 0.0005, 0.0002, 0.0003, 0.0013, 0.004, 0.0001, 0.0001, 0.0001, 0.0001, 0.0002, 0.0003, 0.0002, 0.0004, 0.0002, 0.0006, 0.0, 0.0003, 0.0001, 0.0002, 0.0096, 0.11, 0.073, 0.0, 0.0, 0.0, 0.0006, 0.0003, 0.0001, 0.0005, 0.0003, 0.0002, 0.0001, 0.0001, 0.0, 0.0002, 0.0111, 0.1245, 0.1558, 0.0003, 0.0003, 0.0002, 0.0004, 0.0, 0.0001, 0.0002, 0.0002, 0.0004, 0.0002, 0.0004, 0.0001, 0.0003, 0.0151, 0.0282, 0.0752, 0.0097, 0.0001, 0.0001, 0.0001, 0.0004, 0.0008, 0.0003, 0.0003, 0.0002, 0.0001, 0.0009, 0.0002, 0.0004, 0.0002, 0.0177, 0.0085, 0.0002, 0.0005, 0.0001, 0.0002, 0.0001, 0.0005, 0.0001, 0.0008, 0.0003, 0.0002, 0.0001, 0.0001, 0.0009, 0.0005, 0.0254, 0.0261, 0.0001, 0.0002, 0.0004, 0.0002, 0.0002, 0.0001, 0.0001, 0.0006, 0.0002, 0.0008, 0.0002, 0.0002, 0.0003, 0.0001, 0.0004, 0.0109, 0.0063, 0.0004, 0.0007, 0.0006, 0.0002, 0.0003, 0.0002, 0.0013, 0.0001, 0.0001, 0.0004, 0.0002, 0.0001, 0.0002, 0.0003, 0.0003, 0.0001, 0.0001, 0.0, 0.0006, 0.0004, 0.0005, 0.0004, 0.0005, 0.0004, 0.0002, 0.0001, 0.0002, 0.0002, 0.0005, 0.0009, 0.0005, 0.0, 0.0007, 0.0004, 0.001, 0.0003, 0.0001, 0.0012, 0.0003, 0.0002, 0.0001, 0.0002, 0.0002, 0.0005, 0.0002, 0.0001, 0.0009, 0.0001, 0.0001, 0.0008, 0.0009, 0.0007, 0.0004, 0.0001, 0.0004, 0.0004, 0.0002, 0.0006, 0.0008, 0.0005, 0.0003, 0.0002, 0.0006, 0.0015, 0.0001, 0.0003, 0.0004, 0.0013, 0.0003, 0.0001, 0.0008, 0.0004, 0.0003, 0.0002, 0.0003, 0.0001, 0.0001, 0.0001, 0.0001, 0.0002, 0.0001, 0.0003, 0.0002, 0.0004, 0.001, 0.0006], [0.0015, 0.0003, 0.0005, 0.0018, 0.0002, 0.0018, 0.0007, 0.0011, 0.0007, 0.0017, 0.0008, 0.0006, 0.0005, 0.0004, 0.0007, 0.0012, 0.0007, 0.0007, 0.0011, 0.0033, 0.0013, 0.0001, 0.0013, 0.0018, 0.0001, 0.0003, 0.0029, 0.0006, 0.0005, 0.0005, 0.0003, 0.0002, 0.001, 0.0002, 0.0008, 0.0005, 0.0005, 0.0002, 0.0008, 0.0019, 0.0001, 0.0012, 0.0015, 0.0011, 0.0002, 0.0003, 0.0009, 0.0003, 0.0009, 0.0023, 0.0013, 0.0037, 0.0004, 0.0023, 0.0006, 0.0007, 0.0003, 0.0008, 0.0004, 0.0006, 0.0007, 0.001, 0.0009, 0.0009, 0.0005, 0.0002, 0.0006, 0.0007, 0.0003, 0.0005, 0.003, 0.0081, 0.0008, 0.0002, 0.0011, 0.0004, 0.0006, 0.003, 0.0007, 0.0004, 0.0012, 0.0031, 0.0001, 0.0003, 0.0003, 0.0006, 0.1658, 0.0712, 0.0048, 0.0002, 0.0001, 0.0003, 0.0012, 0.0007, 0.0003, 0.0007, 0.0031, 0.0017, 0.0004, 0.0004, 0.0001, 0.0008, 0.0022, 0.0049, 0.0105, 0.0012, 0.0007, 0.0008, 0.0006, 0.0002, 0.0002, 0.0004, 0.0006, 0.0007, 0.0016, 0.0004, 0.0001, 0.0007, 0.0083, 0.0625, 0.0198, 0.0086, 0.0005, 0.0002, 0.0004, 0.002, 0.0034, 0.0015, 0.0003, 0.0007, 0.0006, 0.0032, 0.0005, 0.0008, 0.0006, 0.0667, 0.0396, 0.0007, 0.0037, 0.0005, 0.0004, 0.0002, 0.0004, 0.0004, 0.0016, 0.0009, 0.0017, 0.0002, 0.0005, 0.0044, 0.0031, 0.0478, 0.0678, 0.0005, 0.0007, 0.0006, 0.0004, 0.0015, 0.0002, 0.0005, 0.0017, 0.0005, 0.0031, 0.0004, 0.0013, 0.0003, 0.0002, 0.0006, 0.0345, 0.0207, 0.0008, 0.0008, 0.0008, 0.0024, 0.0011, 0.0002, 0.0039, 0.0003, 0.0008, 0.0006, 0.0008, 0.0002, 0.0005, 0.0004, 0.0017, 0.0003, 0.0004, 0.0002, 0.0015, 0.0016, 0.0011, 0.0005, 0.0011, 0.0004, 0.0005, 0.0019, 0.0006, 0.002, 0.0012, 0.0012, 0.0011, 0.0003, 0.0025, 0.0009, 0.0017, 0.0006, 0.0008, 0.0012, 0.0004, 0.0006, 0.0006, 0.0009, 0.0004, 0.0011, 0.0009, 0.0005, 0.0008, 0.0002, 0.0002, 0.0015, 0.0009, 0.0008, 0.0003, 0.0005, 0.0008, 0.0016, 0.0005, 0.0025, 0.0014, 0.0013, 0.002, 0.0004, 0.001, 0.0012, 0.0003, 0.001, 0.0009, 0.0015, 0.0003, 0.0013, 0.0014, 0.0014, 0.0021, 0.0013, 0.0019, 0.0007, 0.0004, 0.0008, 0.0008, 0.0014, 0.0002, 0.0012, 0.0007, 0.0008, 0.0028, 0.0013]])).max()), 0, tol=0.03)
fig, ax = plt.subplots(2, 7, figsize=(11, 3.4))
for row, k in enumerate((0, 4)):
    ax[row, 0].imshow(ims[k]); ax[row, 0].set_title(PHOTOS[k], fontsize=8)
    for h in range(6):
        ax[row, h + 1].imshow(cls[k, h].reshape(16, 16), cmap="gray_r"); ax[row, h + 1].set_title(f"head {h + 1}", fontsize=8)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle("Where the summary token looks, last block (ink: attention)", fontsize=10); plt.show()

Darcet et al. (2023) found that large ViTs grow a few high-norm *artefact* patches that grab attention, and that four
extra **register** tokens soak it up. In this small model at 224 pixels the lecture found no artefacts (no patch with
more than 3 times the median length at the last block); the register model's summary token still sends some attention
to its registers.

In [ ]:
norms = o.hidden_states[-1][:, 1:].norm(dim=-1).cpu().numpy()
artefacts = int(sum((p > 3 * np.median(p)).sum() for p in norms))
print(f"patches longer than 3 x the median, over the 8 photos: {artefacts}")
check("artefact patches", artefacts, 0)
with torch.no_grad():
    o_r = vit["facebook/dinov2-with-registers-small"](pixel_values=X, output_attentions=True)
to_reg = o_r.attentions[-1][:, :, 0, 1:5].sum(-1).mean().item()
print(f"with registers: the summary token's share on its 4 registers, last block, 8 photos: {to_reg:.3f} "
      f"(the lecture, over 30 photos: {0.2026})")

## Your turn

Each exercise has a one-line hint. Worked solutions are at the end.

**1. The divisor.** Block 11, head 12 divides by 8. What share does *for* get if it divided by 1, or by 32?
*Hint: `torch.softmax(qk / d, -1)[11]`.*

In [ ]:
# your code here

**2. Two words at once.** Hide *we* and *went* together; then *went* and *for*.
*Hint: the list `hide` from Step 2: bit j stands for word j.*

In [ ]:
# your code here

**3. The head that pushes walk hardest.** Block 12, head 4. Where does the last word look, and what would it push from
that word?
*Hint: `c["pattern"][11, 3, -1]`, and Step 10's `each` and `read` with block 12's weights.*

In [ ]:
# your code here

**4. The resting head.** Switching block 6, head 2 off *raises* walk. Which words gain the most?
*Hint: compare `log` chances with and without `zero_heads=[(5, 1)]`.*

In [ ]:
# your code here

**5. Your own bigram.** In the 100 million FineWeb tokens of Part 4, what follows *Los*? And *according*?
*Hint: Step 18's `follows()`.*

In [ ]:
# your code here

## Solutions

**1. The divisor.**

In [ ]:
for d in (1, 8, 32):
    s = torch.softmax(qk / d, -1)
    print(f"divide by {d:2}: 'for' {100 * s[11]:5.1f}%, 'It' {100 * s[0]:5.1f}%")
check("divide by 8: 'for'", torch.softmax(qk / 8, -1)[11].item(), 0.85341, tol=1e-4)
print("A small divisor makes the head pick one word; a large one spreads it out.")

**2. Two words at once.**

In [ ]:
print(f"we + went hidden: walk {100 * hide[(1 << 9) | (1 << 10)]:.2f}%;  went + for hidden: walk {100 * hide[(1 << 10) | (1 << 11)]:.2f}%")
check("we + went hidden: walk", hide[(1 << 9) | (1 << 10)], 0.00145, tol=5e-5)
check("went + for hidden: walk", hide[(1 << 10) | (1 << 11)], 0.00382, tol=5e-5)

**3. The head that pushes walk hardest.**

In [ ]:
pl = c["pattern"][11, 3, -1]
print("block 12, head 4 looks at:", [(WORDS[j], round(pl[j].item(), 2)) for j in pl.topk(3).indices.tolist()])
Wb = SD["h.11.attn.c_attn.weight"]; Bb = SD["h.11.attn.c_attn.bias"]
x12 = layer_norm(c["resid"][11], SD["h.11.ln_1.weight"], SD["h.11.ln_1.bias"])
each12 = (x12 @ Wb[:, 1536 + 64 * 3: 1536 + 64 * 4] + Bb[1536 + 64 * 3: 1536 + 64 * 4]) @ SD["h.11.attn.c_proj.weight"][64 * 3: 64 * 4]
r12 = ((each12 - each12.mean(-1, keepdim=True)) / sigma * gain) @ WTE.T
j = int(pl.argmax())
print(f"from {WORDS[j]!r} it would push:", [decode([i]) for i in (r12[j] - r12[j].mean()).topk(6).indices.tolist()])
check("its push on walk", push(c["head_out"][11, 3, -1]), 1.0517, tol=3e-3)

**4. The resting head.**

In [ ]:
base = torch.log_softmax(c["logits"][-1], -1)
off = torch.log_softmax(run(ids, zero_heads=[(5, 1)])["logits"][-1], -1)
gain = (off.exp() - base.exp()).topk(5)
print("gain the most chance:", [(decode([j]), f"+{100 * v:.1f}%") for v, j in zip(gain.values.tolist(), gain.indices.tolist())])
top3 = [decode([j]) for j in off.topk(3).indices.tolist()]
print("the top three now:", top3)
check("the top three words with block 6, head 2 off", top3, [' walk', ' cold', ' warm'])
print("Its usual job seems to be keeping the weather words down; off, cold and warm rise, and so does walk.")

**5. Your own bigram.**

In [ ]:
for w in (" Los", " according"):
    top, n_ = follows(w)
    print(f"after {w!r} ({n_:,} times): {top}")
    check(f"what most often follows {w!r}", top[0][0], {' United': {'top': [' States', 0.7081], 'n': 17531}, ' Barack': {'top': [' Obama', 0.9419], 'n': 1050}, ' New': {'top': [' York', 0.3673], 'n': 35089}, ' the': {'top': [' same', 0.0103], 'n': 3413372}, ' Mr': {'top': ['.', 0.7576], 'n': 10618}, ' according': {'top': [' to', 0.9874], 'n': 11669}, ' Los': {'top': [' Angeles', 0.825], 'n': 3937}, ' per': {'top': [' cent', 0.1416], 'n': 24162}, ' walk': {'top': [' in', 0.0675], 'n': 9221}, ' for': {'top': [' the', 0.153], 'n': 743923}}[w]["top"][0])

In [ ]:
print(f"The whole notebook ran in {(time.time() - T0) / 60:.1f} minutes.")